# Bottle Base Inspection — Binary Classifier
**Goal:** Predict whether a bottle is `GOOD (0)` or `FAULTY (1)` from an image of its base.

**Target:** F1 > 98% on the FAULTY class, with recall ≥ 99%.

---
## Architecture Overview
```
CSV (label + area_px)
  → Label Resolver       (3-tier logic → binary 0/1)
  → ROI Extractor        (Hough circle → crop → CLAHE → 224×224)
  → Augmentation         (Albumentations, train only)
  → EfficientNet-B0      (ImageNet pretrained backbone)
  → Classification Head  (Dropout → Linear → BN → ReLU → Linear(1))
  → Focal Loss           (γ=2, α=0.75 — hard-example focus)
  → Two-Phase Training   (freeze backbone → warm head → unfreeze → cosine LR)
  → Threshold Calibration(PR curve sweep on val, safety gate recall ≥ 0.99)
  → Evaluation           (F1, AUC, confusion matrix, plots)
```

## 0 — Install & Imports

In [1]:
# ── Install missing packages (Kaggle already has torch/timm/albumentations) ──

import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('installing packages...')
pip_install('timm')
pip_install('albumentations')
pip_install('optuna')
print('installation completed')

installing packages...
installation completed


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# IMPORTS — all third-party dependencies in one place
# ═══════════════════════════════════════════════════════════════════════════
print("Importing python packages..")
# Standard library
import os, math, json, logging, warnings, copy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Numerical / data
import numpy as np
import pandas as pd

# Image processing
import cv2
from PIL import Image

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Model zoo
import timm

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
)

# Hyperparameter search
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Visualisation
import matplotlib
matplotlib.use('Agg')   # safe for notebook + server both
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
from tqdm.auto import tqdm

print("Done importing")

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger(__name__)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Importing python packages..


/home/dll0706/Documents/Bakwowi_Junior_CV_Project/.conda_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Done importing
Device: cuda
PyTorch: 2.12.0+cu130


In [26]:
# import shutil
# shutil.rmtree("/kaggle/working/")
# os.remove('/kaggle/working/submission.csv')
# shutil.copy('/kaggle/input/competitions/1st-krones-vision-ai-challenge/train.csv', '/kaggle/working/')
# os.mkdir('outputs')
# shutil.copy('/kaggle/input/datasets/bakwowijunior/predictions/predictions.csv', '/kaggle/working/outputs')
# shutil.copy('/kaggle/input/models/bakwowijunior/bottle-model/pytorch/default/1/final_model.pth', '/kaggle/working/')
# shutil.copytree('/kaggle/input/datasets/bakwowijunior/processed-data/', '/kaggle/working/', dirs_exist_ok=True)

'/kaggle/working/train.csv'

In [ ]:
print('hello world')

---
## 1 — Configuration

Everything in one dictionary. Change values here; no hunting through code.

In [3]:
CFG = {
    # ── Paths ─────────────────────────────────────────────────────────────
    'annotation_csv':  './1st-krones-vision-ai-challenge/train.csv',
    'processed_dir':   './1st-krones-vision-ai-challenge/processed',
    'output_dir':      './1st-krones-vision-ai-challenge/outputs',
    'checkpoint_path': './1st-krones-vision-ai-challenge/outputs/best_model.pt',

    # NEW — path to predictions.csv from the PREVIOUS run.
    # First run: file won't exist → falls back to COCO conflict weighting.
    # Every run after: file exists → uses FP/FN prediction-based weighting.
    'predictions_csv': './1st-krones-vision-ai-challenge/predictions.csv',

    # ── Label columns ─────────────────────────────────────────────────────
    'label_col':  'label',
    'area_col':   None,
    'image_col':  'image_path',
    'image_dir':  './1st-krones-vision-ai-challenge/train_images',

    # ── Image / ROI ────────────────────────────────────────────────────────
    'input_size':             320,
    'roi_category_id':        22,
    'fixed_cx':               640,
    'fixed_cy':               512,
    'fixed_radius':           450,
    'hough_dp':               1.2,
    'hough_min_dist':         500,
    'hough_param1':           100,
    'hough_param2':           40,
    'hough_min_r':            300,
    'hough_max_r':            600,
    'roi_margin':             20,
    'clahe_clip':             2.0,
    'clahe_grid':             (8, 8),
    'good_feature_mask_prob': 0.7,
    'img_mean':               [0.485, 0.456, 0.406],
    'img_std':                [0.229, 0.224, 0.225],

    # ── Data splits ────────────────────────────────────────────────────────
    'train_frac': 0.70,
    'val_frac':   0.15,

    # ── Model ─────────────────────────────────────────────────────────────
    'backbone':   'hrnet_w32',
    'dropout1':   0.5,
    'hidden_dim': 512,
    'dropout2':   0.2,

    # ── Loss ──────────────────────────────────────────────────────────────
    'focal_gamma': 3.0,
    'focal_alpha': 0.35,

    # NEW — per-sample loss multipliers for misclassified bottles.
    # fp_weight: GOOD bottles predicted as FAULTY (false positives).
    #   These are the 300-600 bottles reducing precision.
    #   Higher value → model pushed harder to lower their scores.
    # fn_weight: FAULTY bottles predicted as GOOD (false negatives).
    #   Upweighting maintains recall while fp_weight improves precision.
    # hard_example_weight: used on first run (fallback when no predictions.csv).
    'fp_weight':            4.0,
    'fn_weight':            1.5,
    'hard_example_weight':  3.0,

    # ── Training ──────────────────────────────────────────────────────────
    'warmup_epochs':   4,
    'batch_size':      32,   # GPU batch size (replaces per_core_batch_size)
    # 'accumulate_grad_batches': 2,
    'phase1_epochs':  12,
    'phase2_epochs':  18,
    'phase3_epochs':  10,
    'phase4_epochs':  10,
    'total_epochs':   50,
    'lr_head':        1e-3,
    'lr_phase2':      8e-5,
    'lr_phase3_bb':   5e-6,
    'lr_phase3_hd':   3e-5,
    'lr_phase4':      1e-6,
    'min_lr':         1e-7,
    'weight_decay':   5e-4,
    'grad_clip':      1.0,
    'amp':            True,
    'num_workers':    4,

    # ── Early stopping ─────────────────────────────────────────────────────
    'early_stop_patience':  20,
    'early_stop_min_delta': 5e-4,

    # ── Threshold calibration ──────────────────────────────────────────────
    'min_recall_faulty': 0.990,
    'default_threshold': 0.5,

    # ── Optuna ────────────────────────────────────────────────────────────
    'optuna_n_trials': 30,
    'optuna_n_epochs': 60,
}
print('Config ready.')

Config ready.


---
## 3 — ROI Extraction

### What it does
Extracts the circular bottle base from a raw camera image by detecting the circle, masking everything outside it, and cropping to a tight square.

### Why each step matters
| Step | Why |
|------|-----|
| Median blur | More robust than Gaussian to the salt-and-pepper noise from industrial cameras |
| Hough Circle Transform | Finds the circular bottle base even when the conveyor belt or background is visible |
| Closest-to-centre selection | Handles false positives from reflections or background circles |
| Circular masking | Removes conveyor belt texture that would confuse the model |
| Square padding | Ensures consistent input shape without distorting the circle |
| CLAHE on LAB L-channel | Enhances local contrast specifically in the luminance channel, making contamination-light and glass imperfections dramatically more visible without over-saturating colours |

### Innovation
CLAHE in LAB space (not BGR) is the key technique here. Applying contrast enhancement to the L (lightness) channel only preserves colour information, which is critical for detecting `contamination_dark` vs `contamination_light`. Naive histogram equalisation on BGR would distort the colour signal.

In [4]:
# ── Build a GOOD feature bbox map from COCO ───────────────────────────────
# Maps filename → list of (x, y, w, h) bboxes for GOOD annotations

GOOD_FEATURE_LABELS = {
    'water drop', 'water_drop',
    'foam residue', 'foam_residue',
    'embossing',
    'no fault', 'no_fault',
}

def build_good_feature_mask_map(
    coco:           dict,
    id_to_filename: Dict[int, str],
    id_to_catname:  Dict[int, str],
) -> Dict[str, List[Tuple[int, int, int, int]]]:
    """
    Build a lookup dict: filename → list of (x, y, w, h) bboxes
    for all GOOD annotations on that image.

    Used by GoodFeatureMasker to zero out those regions during training.
    This tells the model: "these regions are irrelevant to the decision."
    """
    mask_map: Dict[str, List] = {}

    for ann in coco['annotations']:
        cat_name = id_to_catname.get(ann['category_id'], '').lower().strip()

        # Only process GOOD feature annotations
        is_good = any(label in cat_name for label in GOOD_FEATURE_LABELS)
        if not is_good:
            continue

        bbox = ann.get('bbox')  # COCO format: [x, y, w, h]
        if not bbox or len(bbox) != 4:
            continue

        fname = Path(id_to_filename.get(ann['image_id'], '')).name
        if fname not in mask_map:
            mask_map[fname] = []
        mask_map[fname].append(tuple(int(v) for v in bbox))

    total_annotations = sum(len(v) for v in mask_map.values())
    log.info(
        'Good-feature mask map: %d images | %d total GOOD annotations to mask',
        len(mask_map), total_annotations,
    )
    return mask_map


class GoodFeatureMasker(A.ImageOnlyTransform):
    """
    Randomly zero out annotated GOOD feature regions during training.

    HOW it works
    ────────────
    For each image, looks up its filename in mask_map to find COCO bboxes
    of GOOD features (water drops, foam, embossing). Applies each bbox as
    a rectangular black mask with probability mask_prob.

    WHY this is effective
    ─────────────────────
    The model cannot use a signal it cannot see. When water drops are masked
    out, the model must base its FAULTY prediction on other features. After
    enough training epochs with partial masking, the model learns to route
    around water drops and foam when making the binary decision.

    The key insight: mask_prob < 1.0 means some images are masked and some
    are not. The model therefore cannot simply learn "when water drop region
    is black → FAULTY". It must learn the underlying defect texture instead.

    Parameters
    ----------
    mask_map  : dict from build_good_feature_mask_map()
    filename  : the current image's basename (injected by BottleDataset)
    mask_prob : probability of masking each individual GOOD bbox (not the
                entire image). 0.7 means each water drop has 70% chance
                of being masked in any given training step.
    fill_value: pixel value to fill masked region (0 = black, matches the
                circular mask border already in the image)
    """
    def __init__(
        self,
        mask_map:   Dict[str, List],
        mask_prob:  float = 0.7,
        fill_value: int   = 0,
        p:          float = 1.0,
        always_apply: bool = False,
    ):
        super().__init__(p=p, always_apply=always_apply)
        self.mask_map   = mask_map
        self.mask_prob  = mask_prob
        self.fill_value = fill_value
        # filename is injected at call time via additional_targets
        self._current_filename = None

    def set_filename(self, filename: str):
        """Called by BottleDataset before each augmentation call."""
        self._current_filename = filename

    def apply(self, img: np.ndarray, **params) -> np.ndarray:
        fname = self._current_filename
        if fname is None or fname not in self.mask_map:
            return img  # no GOOD annotations for this image

        result = img.copy()
        h, w   = img.shape[:2]

        for (bx, by, bw, bh) in self.mask_map[fname]:
            # Mask this bbox with mask_prob probability
            if np.random.random() > self.mask_prob:
                continue  # keep this feature visible this time

            # Clip to image bounds (COCO coords can slightly exceed image)
            x1 = max(0, int(bx))
            y1 = max(0, int(by))
            x2 = min(w, int(bx + bw))
            y2 = min(h, int(by + bh))

            result[y1:y2, x1:x2] = self.fill_value

        return result

    def get_transform_init_args_names(self) -> tuple:
        return ('mask_prob', 'fill_value')

In [5]:
def load_coco_roi_map(
    annotation_path: str = './1st-krones-vision-ai-challenge/train_annotations.json',
    roi_category_id: int = 22,
) -> Dict[str, Tuple[int, int, int]]:
    """
    Parse a COCO annotations.json and build a lookup dict:
        filename (basename only) → (cx, cy, radius)
 
    Only annotations whose category_id matches roi_category_id are used.
    Everything else in the file (defect annotations, etc.) is ignored.
 
    Parameters
    ----------
    annotation_path : path to annotations.json
    roi_category_id : category_id that marks the bottle base ROI (default 22)
 
    Returns
    -------
    dict  { 'bottle_001.png': (1024, 1024, 900), ... }
 
    Why basename only?
    ──────────────────
    COCO file_name can be 'train/bottle_001.png' or just 'bottle_001.png'.
    Storing only the basename makes the map work regardless of where raw
    images are stored on disk at training time.
    """
    with open('./1st-krones-vision-ai-challenge/train_annotations.json', 'r') as f:
        coco = json.load(f)
 
    # image_id → basename
 
    id_to_filename = {img['id']: img['file_name'] for img in coco['images']}
    id_to_catname  = {cat['id']: cat['name']      for cat in coco['categories']}
 
    roi_map: Dict[str, Tuple[int, int, int]] = {}
    skipped = 0
 
    for ann in coco['annotations']:
        if ann['category_id'] != roi_category_id:
            continue   # skip defect annotations, only want ROI
 
        filename = id_to_filename.get(ann['image_id'])
        if filename is None:
            continue
 
        # Prefer segmentation polygon (more accurate circle fit)
        # Fall back to bbox if no segmentation
        if ann.get('segmentation') and len(ann['segmentation']) > 0:
            try:
                cx, cy, radius = _circle_from_segmentation(ann['segmentation'])
            except Exception:
                if ann.get('bbox') and len(ann['bbox']) == 4:
                    cx, cy, radius = _circle_from_bbox(ann['bbox'])
                else:
                    skipped += 1
                    continue
        elif ann.get('bbox') and len(ann['bbox']) == 4:
            cx, cy, radius = _circle_from_bbox(ann['bbox'])
        else:
            skipped += 1
            continue
 
        roi_map[filename] = (cx, cy, radius)
 
    log.info(
        'COCO ROI map — %d images mapped | category_id=%d | skipped=%d',
        len(roi_map), roi_category_id, skipped,
    )
 
    if len(roi_map) == 0:
        raise ValueError(
            f'No annotations found for category_id={roi_category_id}. '
            f'Check annotation_json path and roi_category_id in CFG.\n'
            f'Available category ids: '
            f'{[c["id"] for c in coco["categories"]]}\n'
            f'Available category names: '
            f'{[c["name"] for c in coco["categories"]]}'
        )
 
    return roi_map, coco, id_to_filename, id_to_catname
 
 
def _circle_from_bbox(bbox: list) -> Tuple[int, int, int]:
    """
    Convert COCO bbox [x, y, w, h] to (cx, cy, radius).
 
    COCO uses top-left corner format. Centre is the midpoint.
    Radius is half the shorter side — stays inside the bbox for any
    aspect ratio, which handles slightly non-square bottle annotations.
    """
    x, y, w, h = bbox
    cx     = int(x + w / 2)
    cy     = int(y + h / 2)
    radius = int(min(w, h) / 2)
    return cx, cy, radius
 
 
def _circle_from_segmentation(segmentation: list) -> Tuple[int, int, int]:
    """
    Fit the minimum enclosing circle to a COCO segmentation polygon.
 
    COCO segmentation format: [[x1, y1, x2, y2, x3, y3, ...]]
    Uses the first (largest) polygon if multiple exist.
 
    cv2.minEnclosingCircle gives the exact smallest circle containing all
    polygon vertices — more accurate than bbox-derived radius when the
    annotator drew a polygon rather than a perfect square bbox.
    """
    flat = segmentation[0]   # first polygon: [x1, y1, x2, y2, ...]
    pts  = np.array(flat, dtype=np.float32).reshape(-1, 1, 2)
    (cx, cy), radius = cv2.minEnclosingCircle(pts)
    return int(cx), int(cy), int(radius)
 
 
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 — Validate before training (catches missing annotations early)
# ═══════════════════════════════════════════════════════════════════════════
 
def validate_roi_map(
    df: pd.DataFrame,
    roi_map: Dict[str, Tuple[int, int, int]],
    image_col: str = 'image_path',
) -> None:
    """
    Check every image in the DataFrame has a COCO ROI annotation.
    Logs warnings for missing entries so you know before training starts
    which images will fall back to the fixed cx/cy/radius.
    """
    missing = [
        Path(p).name
        for p in df[image_col]
        if Path(p).name not in roi_map
    ]
 
    if missing:
        log.warning(
            '%d / %d images have no ROI annotation → will use fixed fallback. '
            'First 10: %s',
            len(missing), len(df), missing[:10],
        )
    else:
        log.info(
            'ROI validation passed — all %d images have COCO annotations.', len(df)
        )
 
 
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 — CLAHE and padding helpers (unchanged from original)
# ═══════════════════════════════════════════════════════════════════════════
 
def apply_clahe(bgr: np.ndarray, clip_limit: float, tile_grid: tuple) -> np.ndarray:
    """
    Apply CLAHE to the L channel in LAB colour space.
    Enhances local contrast without shifting hues — critical for
    contamination_light and glass_imperfection which are subtle brightness
    differences from the clean glass background.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    lab   = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq  = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)
 
 
def pad_to_square(img: np.ndarray) -> np.ndarray:
    """
    Pad the shorter axis with black pixels to produce a square image.
    Black blends with the circular mask border already in the image.
    """
    h, w = img.shape[:2]
    if h == w:
        return img
    side   = max(h, w)
    canvas = np.zeros((side, side, 3), dtype=img.dtype)
    y_off  = (side - h) // 2
    x_off  = (side - w) // 2
    canvas[y_off:y_off + h, x_off:x_off + w] = img
    return canvas
 
 
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4 — Core ROI extraction (COCO-powered, Hough removed)
# ═══════════════════════════════════════════════════════════════════════════
 
def extract_roi_from_array(
    bgr:    np.ndarray,
    cfg:    dict,
    cx:     Optional[int] = None,
    cy:     Optional[int] = None,
    radius: Optional[int] = None,
) -> Image.Image:
    """
    Full ROI pipeline for a single image (BGR numpy array).
 
    When cx/cy/radius are provided (from COCO map), no detection runs.
    When they are None (image missing from COCO map), falls back to the
    fixed centre/radius from CFG.
 
    Pipeline
    ────────
    1. Resolve circle: COCO-derived if available, else fixed fallback
    2. Circular mask  — everything outside bottle base → black
    3. Bounding square crop + margin
    4. Pad to square if the crop is not already square
    5. CLAHE on LAB L-channel
    6. Resize to input_size × input_size (INTER_AREA)
    7. BGR → RGB → PIL Image
 
    Parameters
    ----------
    bgr            : raw camera image as BGR numpy array
    cfg            : CFG dict
    cx, cy, radius : from COCO roi_map; all three or none
    """
    h, w = bgr.shape[:2]
 
    # ── Step 1: resolve circle ────────────────────────────────────────────
    if cx is None or cy is None or radius is None:
        # No COCO annotation for this image — use fixed fallback
        cx     = cfg['fixed_cx']
        cy     = cfg['fixed_cy']
        radius = cfg['fixed_radius']
        log.debug('No COCO ROI — using fixed fallback (cx=%d, cy=%d, r=%d).',
                  cx, cy, radius)
 
    margin = cfg['roi_margin']
 
    # ── Step 2: circular mask ─────────────────────────────────────────────
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), radius + margin // 2, 255, thickness=-1)
    masked = cv2.bitwise_and(bgr, bgr, mask=mask)
 
    # ── Step 3: bounding square crop ─────────────────────────────────────
    x1   = max(cx - radius - margin, 0)
    y1   = max(cy - radius - margin, 0)
    x2   = min(cx + radius + margin, w)
    y2   = min(cy + radius + margin, h)
    crop = masked[y1:y2, x1:x2]
 
    # ── Step 4: square padding ────────────────────────────────────────────
    crop = pad_to_square(crop)
 
    # ── Step 5: CLAHE ─────────────────────────────────────────────────────
    crop = apply_clahe(crop, cfg['clahe_clip'], cfg['clahe_grid'])
 
    # ── Step 6: resize ────────────────────────────────────────────────────
    size = cfg['input_size']
    crop = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)
 
    # ── Step 7: BGR → RGB → PIL ───────────────────────────────────────────
    return Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
 
 
def extract_roi_from_path(
    image_path: str,
    cfg:        dict,
    roi_map:    Optional[Dict[str, Tuple[int, int, int]]] = None,
) -> Image.Image:
    """
    Load an image from disk and extract its ROI.
 
    Looks up the filename in roi_map first. If found, passes the
    COCO-derived circle directly — no detection runs.
    If not found (or roi_map is None), uses the fixed fallback.
 
    Parameters
    ----------
    image_path : full path to raw image file
    cfg        : CFG dict
    roi_map    : output of load_coco_roi_map(); None = always use fallback
    """
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')
 
    cx = cy = radius = None
    if roi_map is not None:
        fname = Path(image_path).name
        if fname in roi_map:
            cx, cy, radius = roi_map[fname]
 
    return extract_roi_from_array(bgr, cfg, cx=cx, cy=cy, radius=radius)
 
 
# ═══════════════════════════════════════════════════════════════════════════
# STEP 5 — Batch preprocessing (saves COCO-cropped images to disk once)
# ═══════════════════════════════════════════════════════════════════════════
 
def preprocess_and_save_all(
    df:         pd.DataFrame,
    output_dir: str,
    cfg:        dict,
    roi_map:    Dict[str, Tuple[int, int, int]],
    image_col:  str = 'image_path',
    n_jobs:     int = 4,
) -> pd.DataFrame:
    """
    Pre-compute and save COCO-cropped ROIs to disk.
    Run ONCE before training. Each epoch then loads the fast pre-cropped
    image directly without re-running any detection.
 
    Returns a modified DataFrame where image_col points to pre-cropped files.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
 
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    errors = []
 
    def _do_one(row):
        src = Path(row[image_col])
        dst = out / src.name
        if dst.exists():
            return str(dst), None
        try:
            pil = extract_roi_from_path(str(src), cfg, roi_map=roi_map)
            pil.save(str(dst))
            return str(dst), None
        except Exception as e:
            return str(dst), str(e)
 
    new_paths = []
    with ThreadPoolExecutor(max_workers=n_jobs) as pool:
        futures = {pool.submit(_do_one, row): i for i, row in df.iterrows()}
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc='Pre-processing ROIs (COCO)'):
            dst, err = fut.result()
            new_paths.append((futures[fut], dst, err))
            if err:
                errors.append((dst, err))
 
    new_paths.sort(key=lambda x: x[0])
    df = df.copy()
    df[image_col] = [p for _, p, _ in new_paths]
 
    log.info('Pre-processing done — saved: %d | errors: %d',
             len(df) - len(errors), len(errors))
    for dst, err in errors[:5]:
        log.warning('  FAILED: %s — %s', Path(dst).name, err)
 
    return df



print('ROI extraction functions ready.')

ROI extraction functions ready.


---
## 4 — Augmentation Pipeline

### What it does
Applies randomised visual transformations **only during training** to make the model robust to real-world variation without needing to physically capture all those variations.

### Why each transform is chosen
| Transform | Physical justification |
|-----------|------------------------|
| ±180° rotation | Bottle bases are rotationally symmetric — there is no 'right way up' |
| H/V flip | Same symmetry reason |
| ColorJitter | Camera exposure fluctuations, lighting drift over the production day |
| GaussianBlur | Simulates lens defocus and camera vibration |
| GaussNoise | Industrial camera sensor noise, especially on older cameras |
| GridDistortion | Simulates subtle lens barrel/pincushion distortion across the image |
| CoarseDropout | Simulates small dust particles on the camera lens or minor occlusions |

### Innovation
Albumentations is used instead of torchvision transforms because it operates on numpy arrays in `uint8` format — the native format of OpenCV images — avoiding a wasteful float32 conversion during training. It is also 3–10× faster than PIL-based transforms for the same operations.

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# AUGMENTATION PIPELINE
# ═══════════════════════════════════════════════════════════════════════════

def build_train_transforms(cfg):
    """
    Revised augmentation pipeline for bottle base inspection.

    Key changes vs v1:
    - CoarseDropout removed (was creating false FAULTY signal on GOOD bottles)
    - ColorJitter tightened (glass inspection is lighting-sensitive)
    - Added RandomShadow to simulate conveyor belt edge shadows
    - Added sharpen to help the model distinguish scuffing textures
    - GridDistortion kept but probability reduced
    - GaussNoise kept — sensor noise is real and uniform
    """
    size = cfg['input_size']
    return A.Compose([
        A.Resize(size, size),

        # ── Geometric — unchanged, bottle bases are fully symmetric ──────
        A.Rotate(limit=180, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        # A.RandomResizedCrop(
        #     size=(size, size),
        #     scale=(0.88, 1.0),   # slightly tighter than before
        #     ratio=(0.97, 1.03),  # more square — bottle base is circular
        #     p=0.4,
        # ),

        # ── Photometric — tightened ───────────────────────────────────────
        # A.ColorJitter(
        #     brightness=0.10,   # was 0.20 — reduced to avoid fake contamination
        #     contrast=0.10,     # was 0.20
        #     saturation=0.05,   # was 0.10
        #     hue=0.01,          # was 0.02
        #     p=0.6,
        # ),

        # Simulate real lighting variation (backlight intensity drift)
        # A.RandomBrightnessContrast(
        #     brightness_limit=0.08,
        #     contrast_limit=0.08,
        #     p=0.4,
        # ),
        
        A.RandomGamma(gamma_limit=(75, 125)),
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2
        ),
        

        # ── NEW: Sharpen — helps distinguish scuffing texture ─────────────
        # Scuffing and glass_imperfection are fine-texture defects.
        # Randomly sharpening training images teaches the model to look
        # at texture, not just blob shapes.
        A.Sharpen(
            alpha=(0.1, 0.3),
            lightness=(0.9, 1.1),
            p=0.3,
        ),

        # ── Blur / noise — kept, slightly tightened ───────────────────────
        A.GaussianBlur(blur_limit=3, sigma_limit=(0.1, 1.0), p=0.15),
        A.GaussNoise(var_limit=(3.0, 15.0), p=0.2),  # reduced variance

        # ── Structural — kept but reduced probability ─────────────────────
        A.GridDistortion(
            num_steps=5,
            distort_limit=0.10,  # was 0.15
            p=0.15,              # was 0.20
        ),

        # ── REMOVED: CoarseDropout ────────────────────────────────────────
        # Was creating false dark patches on GOOD bottles → pushed scores
        # toward FAULTY → caused most of the 1,612 false positives

        # ── Always last ───────────────────────────────────────────────────
        A.Normalize(mean=cfg['img_mean'], std=cfg['img_std']),
        ToTensorV2(),
    ])


def build_eval_transforms(cfg: dict) -> A.Compose:
    """
    Build the VALIDATION and TEST augmentation pipeline.
    No randomness — only resize and normalise.

    This is intentionally minimal so that validation metrics are deterministic
    and reproducible. Any randomness here would make val_f1 noisy and
    confuse the early-stopping logic.
    """
    size = cfg['input_size']
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=cfg['img_mean'], std=cfg['img_std']),
        ToTensorV2(),
    ])


print('Augmentation pipelines ready.')

Augmentation pipelines ready.


---
## 5 — Dataset & DataLoader

### What it does
- `BottleDataset` maps a DataFrame row → (image tensor, label tensor) via the ROI extractor and augmentation pipeline
- `make_dataloaders` splits the data, builds a `WeightedRandomSampler` for the training set, and returns ready-to-use DataLoaders

### Class imbalance — two strategies working together
1. **`WeightedRandomSampler`**: each training epoch is drawn with replacement so that GOOD and FAULTY bottles appear approximately equally often. This prevents the model from learning 'always predict GOOD'
2. **Focal Loss** (Stage 6): even within a balanced batch, the loss down-weights easy examples, making the model focus on hard borderline cases

In [ ]:
# def build_targeted_hard_weights(
#     df, coco, id_to_filename, id_to_catname,
#     hard_weight=5.0, image_col='image_path'
# ):
#     """
#     Weight ONLY images that have contamination_dark co-occurring with
#     water_drop or foam_residue. These are the bottles the model confuses.
#     Regular scuffing-only images are NOT weighted — they caused noise.
#     """
#     # Find images with contamination_dark
#     contam_fnames = set()
#     for ann in coco['annotations']:
#         cat = id_to_catname.get(ann['category_id'], '').lower()
#         if 'contamination' in cat and 'dark' in cat:
#             contam_fnames.add(Path(id_to_filename.get(ann['image_id'], '')).name)

#     # Find images with water_drop or foam_residue (GOOD visual signals)
#     good_signal_fnames = set()
#     for ann in coco['annotations']:
#         cat = id_to_catname.get(ann['category_id'], '').lower()
#         if 'water' in cat or 'foam' in cat:
#             good_signal_fnames.add(Path(id_to_filename.get(ann['image_id'], '')).name)

#     # Only weight bottles with BOTH: contamination_dark AND good visual signals
#     # These are the exact conflict cases the model struggles with
#     conflict_fnames = contam_fnames & good_signal_fnames

#     log.info('Targeted conflict weighting: %d images have contamination_dark + water/foam',
#              len(conflict_fnames))

#     df = df.copy()
#     df['_fname'] = df[image_col].apply(lambda x: Path(x).name)
#     df['sample_weight'] = df['_fname'].apply(
#         lambda f: float(hard_weight) if f in conflict_fnames else 1.0
#     )
#     df.drop(columns=['_fname'], inplace=True)
#     return df

In [ ]:
# def build_combined_weights(df, coco, id_to_filename, id_to_catname,
#                              pred_csv, cfg):
#     # Stage 1: targeted conflict weighting (contamination_dark + water_drop)
#     df = build_targeted_hard_weights(
#         df, coco, id_to_filename, id_to_catname,
#         hard_weight=4.0, image_col=cfg['image_col']
#     )
#     # Stage 2: boost further any bottles the model specifically failed on
#     if Path(pred_csv).exists():
#         preds = pd.read_csv(pred_csv)
#         hard_fnames = set(
#             preds.loc[
#                 (preds['true_binary'] == 1) &
#                 (preds['prob_faulty'] < 0.50),
#                 'image_path'
#             ].apply(lambda x: Path(x).name)
#         )
#         df['_fname'] = df[cfg['image_col']].apply(lambda x: Path(x).name)
#         # Multiply: already-weighted conflict images get 4×6=24 weight
#         # Previously unweighted hard images get ×6
#         df['sample_weight'] = df.apply(
#             lambda row: row['sample_weight'] * 6.0
#             if row['_fname'] in hard_fnames else row['sample_weight'],
#             axis=1
#         )
#         df.drop(columns=['_fname'], inplace=True)
#         n_boosted = (df['sample_weight'] > 1.0).sum()
#         log.info('Combined weights: %d images with weight>1.0', n_boosted)
#     return df

In [7]:
# Keep build_targeted_hard_weights exactly as-is — it's used as a fallback.
# build_combined_weights is REPLACED by build_prediction_based_weights below.

def build_coco_fallback_weights(
    df, coco, id_to_filename, id_to_catname,
    hard_weight=3.0, image_col='image_path',
):
    """
    FALLBACK for first run (no predictions.csv yet).
    Weights images where contamination_dark co-occurs with water_drop/foam.
    More targeted than the old broad scuffing + contamination weighting.
    Replaces build_targeted_hard_weights + build_combined_weights.
    """
    contam_fnames = set()
    for ann in coco['annotations']:
        cat = id_to_catname.get(ann['category_id'], '').lower()
        if 'contamination' in cat and 'dark' in cat:
            contam_fnames.add(Path(id_to_filename.get(ann['image_id'], '')).name)

    good_signal_fnames = set()
    for ann in coco['annotations']:
        cat = id_to_catname.get(ann['category_id'], '').lower()
        if 'water' in cat or 'foam' in cat or 'emboss' in cat:
            good_signal_fnames.add(Path(id_to_filename.get(ann['image_id'], '')).name)

    # Only bottles with BOTH contamination_dark AND a GOOD co-annotation
    conflict_fnames = contam_fnames & good_signal_fnames

    df = df.copy()
    df['_fname'] = df[image_col].apply(lambda x: Path(x).name)
    df['sample_weight'] = df['_fname'].apply(
        lambda f: float(hard_weight) if f in conflict_fnames else 1.0
    )
    df.drop(columns=['_fname'], inplace=True)
    log.info('FALLBACK weights — conflict images: %d (×%.1f)', len(conflict_fnames), hard_weight)
    return df


def build_prediction_based_weights(
    df, cfg, coco, id_to_filename, id_to_catname,
    image_col='image_path',
):
    """
    Prediction-based sample weight builder — the main weighting function.

    FIRST RUN (no predictions.csv):
        Falls back to COCO conflict weighting.

    SUBSEQUENT RUNS (predictions.csv from previous run exists):
        Loads the previous run's predictions and assigns weights based on
        which bottles were misclassified:

        FALSE POSITIVES — GOOD bottles predicted as FAULTY:
            Weight = fp_weight (default 6.0)
            These are the 300-600 bottles reducing your precision.
            Effect: model gets 6× penalty → scores pushed below threshold
                    → threshold rises → precision improves.

        FALSE NEGATIVES — FAULTY bottles predicted as GOOD:
            Weight = fn_weight (default 4.0)
            Effect: model gets 4× penalty → scores pushed above threshold
                    → recall maintained while precision improves.

        TRUE POSITIVES / TRUE NEGATIVES:
            Weight = 1.0 — already correct, standard training.

    WHY target FP AND FN simultaneously:
        Upweighting only FP (GOOD bottles) makes the model more conservative.
        Without FN upweighting, recall can drift down. Both together keep the
        model pushing GOOD scores down AND FAULTY scores up simultaneously.
    """
    pred_csv  = cfg.get('predictions_csv', '')
    fp_weight = cfg.get('fp_weight',  6.0)
    fn_weight = cfg.get('fn_weight',  4.0)

    # ── No predictions yet: use COCO fallback ─────────────────────────────
    if not pred_csv or not Path(pred_csv).exists():
        log.info('No predictions.csv found — using COCO fallback weights (first run).')
        return build_coco_fallback_weights(
            df, coco, id_to_filename, id_to_catname,
            hard_weight=cfg.get('hard_example_weight', 3.0),
            image_col=image_col,
        )

    # ── Load previous predictions ─────────────────────────────────────────
    preds = pd.read_csv(pred_csv)
    preds['filename'] = preds['image_path'].apply(lambda x: Path(x).name)

    prev_threshold = float(preds['threshold_used'].iloc[0]) \
        if 'threshold_used' in preds.columns else 0.5

    # FALSE POSITIVES: GOOD (true=0) predicted as FAULTY (pred=1)
    fp_fnames = set(preds.loc[
        (preds['true_binary'] == 0) & (preds['pred_binary'] == 1), 'filename'
    ])

    # FALSE NEGATIVES: FAULTY (true=1) predicted as GOOD (pred=0)
    fn_fnames = set(preds.loc[
        (preds['true_binary'] == 1) & (preds['pred_binary'] == 0), 'filename'
    ])

    log.info(
        'Prediction-based weights | threshold=%.4f | '
        'FP→upweight ×%.1f: %d bottles | FN→upweight ×%.1f: %d bottles',
        prev_threshold, fp_weight, len(fp_fnames), fn_weight, len(fn_fnames),
    )

    df = df.copy()
    df['_fname'] = df[image_col].apply(lambda x: Path(x).name)

    def _assign(fname):
        # FP takes priority over FN when a filename appears in both
        if fname in fp_fnames: return float(fp_weight)
        if fname in fn_fnames: return float(fn_weight)
        return 1.0

    df['sample_weight'] = df['_fname'].apply(_assign)
    df.drop(columns=['_fname'], inplace=True)

    log.info(
        'Weight dist — FP (×%.1f): %d | FN (×%.1f): %d | normal: %d',
        fp_weight, (df['sample_weight'] == fp_weight).sum(),
        fn_weight, (df['sample_weight'] == fn_weight).sum(),
        (df['sample_weight'] == 1.0).sum(),
    )
    return df


print('Sample weight functions ready.')

Sample weight functions ready.


In [8]:
def build_defect_patch_library(
    df:             pd.DataFrame,
    coco:           dict,
    id_to_filename: Dict[int, str],
    id_to_catname:  Dict[int, str],
    cfg:            dict,
    image_col:      str = 'image_path',
    max_patches:    int = 1,
) -> List[np.ndarray]:
    """
    Extract actual contamination_dark image patches from COCO bboxes.
    Used by DefectCutMix to paste real defect regions onto GOOD bottles.
    
    Why real patches rather than synthetic ones?
    The model has already been exposed to SyntheticGoodFeatures (fake water drops).
    Using real contamination_dark patches from actual FAULTY bottles means the
    model trains on the exact visual texture it needs to detect in production.
    """
    patches = []
    fname_to_path = {Path(p).name: p for p in df[image_col]}

    for ann in coco['annotations']:
        cat = id_to_catname.get(ann['category_id'], '').lower()
        if 'contamination' not in cat or 'dark' not in cat:
            continue
        bbox = ann.get('bbox')
        if not bbox or len(bbox) != 4:
            continue

        fname = Path(id_to_filename.get(ann['image_id'], '')).name
        img_path = fname_to_path.get(fname)
        if not img_path or not Path(img_path).exists():
            continue

        bgr = cv2.imread(str(img_path))
        if bgr is None:
            continue

        x, y, w, h = [int(v) for v in bbox]
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(bgr.shape[1], x + w), min(bgr.shape[0], y + h)

        if (x2 - x1) < 5 or (y2 - y1) < 5:
            continue

        patch = cv2.cvtColor(bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
        patches.append(patch)

        if len(patches) >= max_patches:
            break

    log.info('Defect patch library: %d contamination_dark patches extracted', len(patches))
    return patches


class DefectCutMix(A.ImageOnlyTransform):
    """
    Paste a real contamination_dark patch onto the training image.
    
    Applied only to FAULTY bottles (label=1) — it would be incorrect to paste
    defect regions onto GOOD bottles. BottleDataset passes the label so we
    can make this decision per-sample.
    
    For GOOD bottles (label=0): pass-through unchanged.
    For FAULTY bottles (label=1): with probability p, paste 1-2 random
      contamination_dark patches at random locations within the bottle circle.
    
    Effect: the model sees FAULTY bottles where contamination_dark is prominent
    and clearly placed, teaching it to detect contamination regardless of
    surrounding water drops or foam.
    """
    def __init__(
        self,
        patches:    List[np.ndarray],
        patch_scale: Tuple[float, float] = (0.03, 0.08),
        p:          float = 0.3,
        always_apply: bool = False,
    ):
        super().__init__(p=p, always_apply=always_apply)
        self.patches     = patches
        self.patch_scale = patch_scale
        self._is_faulty  = False   # set by BottleDataset before each call

    def set_label(self, is_faulty: bool):
        self._is_faulty = is_faulty

    def apply(self, img: np.ndarray, **params) -> np.ndarray:
        if not self._is_faulty or not self.patches:
            return img   # only paste defects onto FAULTY images

        result = img.copy()
        h, w   = img.shape[:2]
        n_paste = np.random.randint(1, 3)   # paste 1 or 2 patches

        for _ in range(n_paste):
            patch = self.patches[np.random.randint(len(self.patches))]
            ph, pw = patch.shape[:2]

            # Scale patch to a fraction of image size
            scale    = np.random.uniform(*self.patch_scale)
            new_size = max(8, int(min(h, w) * scale))
            patch_r  = cv2.resize(patch, (new_size, new_size))

            # Random placement (avoiding image edges)
            max_x = max(1, w - new_size)
            max_y = max(1, h - new_size)
            px = np.random.randint(0, max_x)
            py = np.random.randint(0, max_y)

            # Blend the patch (alpha composite — doesn't look artificially sharp)
            # alpha  = np.random.uniform(0.6, 0.9)
            alpha = np.random.uniform(0.35, 0.6)
            region = result[py:py+new_size, px:px+new_size]
            if region.shape == patch_r.shape:
                result[py:py+new_size, px:px+new_size] = (
                    alpha * patch_r + (1 - alpha) * region
                ).astype(np.uint8)

        return result

    def get_transform_init_args_names(self) -> tuple:
        return ('patch_scale',)

In [9]:
# ── Updated BottleDataset ──────────────────────────────────────────────────
# Passes the current filename to GoodFeatureMasker before each augmentation.

class BottleDataset(Dataset):
    def __init__(
        self,
        df,
        transform,
        cfg,
        use_roi=False,
        roi_map=None,
        good_feature_masker=None,   # ← new parameter
        defect_cutmix=None,
    ):
        self.df                  = df.reset_index(drop=True)
        self.transform           = transform
        self.cfg                 = cfg
        self.use_roi             = use_roi
        self.roi_map             = roi_map
        self.good_feature_masker = good_feature_masker  # None for val/test
        self.has_weights         = 'sample_weight' in self.df.columns
        self.defect_cutmix       = defect_cutmix

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = row[self.cfg['image_col']]
        is_faulty = bool(row['binary_label'] == 1)
    
        pil = Image.open(path).convert('RGB')
        np_img = np.array(pil)
    
        if self.defect_cutmix is not None:
            self.defect_cutmix.set_label(is_faulty)
            
        # Tell transforms which class this image belongs to
        if self.good_feature_masker is not None:
            self.good_feature_masker.set_filename(Path(path).name)
    
        tensor = self.transform(image=np_img)['image']
        label  = torch.tensor(row['binary_label'], dtype=torch.float32)
        weight = torch.tensor(
            row['sample_weight'] if self.has_weights else 1.0,
            dtype=torch.float32,
        )
        return tensor, label, weight


# ── Updated build_train_transforms ────────────────────────────────────────
# Accepts an optional GoodFeatureMasker instance.

def build_train_transforms(cfg: dict, masker=None, defect_cutmix=None) -> A.Compose:
    size = cfg['input_size']
    transforms = [
        A.Resize(size, size),
        A.Rotate(limit=180, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomResizedCrop(
            size=(size, size), scale=(0.88, 1.0), ratio=(0.97, 1.03), p=0.4
        ),
    ]

    # Insert COCO masker first — before colour jitter so that the filled
    # regions also get colour-jittered (prevents model detecting black patches
    # as an artefact rather than learning to ignore the region)
    if masker is not None:
        transforms.append(masker)
    if defect_cutmix is not None:
        transforms.append(defect_cutmix)

    transforms += [
        # Synthetic GOOD features — adds fake water drops to ALL images
        # so the model sees them on FAULTY images too
        # SyntheticGoodFeatures(max_drops=5, max_foam_patches=3,
        #                        drop_radius_range=(3, 16), p=0.6),
        A.ColorJitter(
            brightness=0.10, contrast=0.10, saturation=0.05, hue=0.01, p=0.6
        ),
        A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.4),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.9, 1.1), p=0.3),
        A.GaussianBlur(blur_limit=3, sigma_limit=(0.1, 1.0), p=0.25),
        A.GaussNoise(var_limit=(3.0, 15.0), p=0.2),
        A.GridDistortion(num_steps=5, distort_limit=0.10, p=0.15),
        A.Normalize(mean=cfg['img_mean'], std=cfg['img_std']),
        ToTensorV2(),
    ]
    return A.Compose(transforms)


# ── Updated make_dataloaders ───────────────────────────────────────────────
# Accepts mask_map and creates the masker for training only.


def compute_pos_weight(train_df):
    n_good   = (train_df['binary_label'] == 0).sum()
    n_faulty = (train_df['binary_label'] == 1).sum()
    w = torch.tensor([n_good / max(n_faulty, 1)], dtype=torch.float32)
    log.info('pos_weight=%.4f (GOOD=%d, FAULTY=%d)', w.item(), n_good, n_faulty)
    return w


def make_weighted_sampler_gpu(labels):
    """
    Standard GPU WeightedRandomSampler — replaces DistributedWeightedSampler
    which was TPU-only and caused 'per_core_batch_size' KeyError on GPU.

    Assigns each sample a weight inversely proportional to its class frequency.
    Result: each epoch draws a roughly 50/50 GOOD/FAULTY balanced set.
    """
    labels   = np.array(labels)
    counts   = np.bincount(labels)
    weights  = 1.0 / counts
    sample_w = [float(weights[l]) for l in labels]
    return WeightedRandomSampler(
        weights=sample_w,
        num_samples=len(sample_w),
        replacement=True,
    )


def make_dataloaders(
    df,
    cfg,
    use_roi=False,
    roi_map=None,
    mask_map=None,        # ← COCO GOOD-feature mask map (None = masking disabled)
    defect_patches=None
):
    """
    Build train/val/test DataLoaders for GPU training.

    Changes vs previous version
    ───────────────────────────
    1. Uses cfg['batch_size'] (GPU) instead of cfg['per_core_batch_size'] (TPU key).
    2. Uses make_weighted_sampler_gpu (WeightedRandomSampler) instead of
       DistributedWeightedSampler — the TPU version was not GPU-compatible.
    3. Accepts mask_map and passes it to the training BottleDataset.
    4. Val/test datasets never receive mask_map — evaluation must be deterministic.
    """
    val_test_frac    = 1.0 - cfg['train_frac']
    val_frac_of_temp = cfg['val_frac'] / val_test_frac

    train_df, temp_df = train_test_split(
        df, test_size=val_test_frac,
        stratify=df['binary_label'], random_state=SEED,
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=1.0 - val_frac_of_temp,
        stratify=temp_df['binary_label'], random_state=SEED,
    )

    log.info('Split — train:%d val:%d test:%d',
             len(train_df), len(val_df), len(test_df))

    # Build GoodFeatureMasker only for training
    masker = None
    if mask_map is not None:
        masker = GoodFeatureMasker(
            mask_map=mask_map,
            mask_prob=cfg.get('good_feature_mask_prob', 0.7),
            fill_value=0,
            p=1.0,
        )
        log.info('GoodFeatureMasker active (mask_prob=%.2f)',
                 cfg.get('good_feature_mask_prob', 0.7))

    if defect_patches is not None:
        defect_cutmix = DefectCutMix(patches=defect_patches, 
                                     patch_scale=(0.03, 0.08), 
                                     p=0.3, 
                                     always_apply=False,)
        log.info('DefectCutMix active')
        

    # Training dataset: gets masker + augmentation
    # Val/test: eval transforms only, no masker
    train_ds = BottleDataset(
        train_df, build_train_transforms(cfg, defect_cutmix=defect_cutmix),
        cfg, use_roi, roi_map, good_feature_masker=masker, defect_cutmix=defect_cutmix
    )
    val_ds  = BottleDataset(val_df,  build_eval_transforms(cfg), cfg, use_roi, roi_map)
    test_ds = BottleDataset(test_df, build_eval_transforms(cfg), cfg, use_roi, roi_map)

    pos_w   = compute_pos_weight(train_df)
    sampler = make_weighted_sampler_gpu(train_df['binary_label'].tolist())

    bs = cfg['batch_size']   # ← was cfg['per_core_batch_size'] — fixed for GPU
    nw = cfg['num_workers']

    train_loader = DataLoader(
        train_ds, batch_size=bs, sampler=sampler,
        num_workers=nw, pin_memory=True,
        persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=bs * 2, shuffle=False,
        num_workers=nw, pin_memory=True,
        persistent_workers=(nw > 0),
    )
    test_loader = DataLoader(
        test_ds, batch_size=bs * 2, shuffle=False,
        num_workers=nw, pin_memory=True,
        persistent_workers=(nw > 0),
    )

    return train_loader, val_loader, test_loader, pos_w


print('DataLoader functions ready.')

DataLoader functions ready.


---
## 6 — Model Architecture

### What it does
Builds the complete neural network: a pretrained EfficientNet-B0 backbone plus a custom 2-layer classification head.

### Why EfficientNet-B0?
EfficientNet scales depth, width, and resolution simultaneously using a compound coefficient — empirically derived via neural architecture search. B0 is the base with ~5.3M parameters, giving an excellent accuracy/latency tradeoff. For this task it is large enough to model subtle defect patterns but small enough for sub-5ms inference with INT8 quantisation.

### Head design
```
GAP output [B, 1280]
  → Dropout(0.4)           ← regularise before the head
  → Linear(1280 → 256)     ← compress features
  → BatchNorm1d(256)        ← stabilise activations during fine-tuning
  → ReLU                   ← non-linearity
  → Dropout(0.2)            ← secondary regularisation
  → Linear(256 → 1)         ← scalar logit
  → (Sigmoid at inference)  ← P(FAULTY) in [0,1]
```

**Why no sigmoid during training?** `BCEWithLogitsLoss` and `BinaryFocalLoss` both combine the sigmoid and loss computation in a numerically stable way (log-sum-exp trick). Applying sigmoid separately and then computing log would lose precision for extreme logit values.

### Innovation: freeze/unfreeze API
Two functions `freeze_backbone` and `unfreeze_backbone_last_n` allow two-phase training without rewriting the training loop. Phase 1 trains only the head; Phase 2 unfreezes and fine-tunes together.

In [10]:
class GeMPooling(nn.Module):
    """
    Generalized Mean Pooling — replaces Global Average Pooling.

    For a spatial feature map F of shape [B, C, H, W]:
      GeM(F)_c = ( (1/HW) * sum_hw( F_chw ^ p ) ) ^ (1/p)

    p=1    : identical to standard GAP (no amplification)
    p=3    : strong amplification of peaks — good for small defects
    p→inf  : approaches max pooling (only strongest activation per channel)

    p is a learnable parameter initialised to 3.0, allowing the model
    to find the optimal amplification level for this specific task.
    Large p values risk instability — clamp to [1, 10].
    """
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, H, W]
        p   = self.p.clamp(min=1.0, max=10.0)
        out = F.avg_pool2d(
            x.clamp(min=self.eps).pow(p),
            kernel_size=(x.shape[-2], x.shape[-1])
        ).pow(1.0 / p)
        return out.view(out.shape[0], -1)   # [B, C]

In [11]:
class MultiScaleFeatureFusion(nn.Module):
    """
    Fuses features from multiple ConvNeXt stages using GeM pooling on each,
    then projects the concatenation to a fixed embedding dimension.

    Why this works for small defects
    ─────────────────────────────────
    The final ConvNeXt stage (10×10 at 320px input) is too coarse to preserve
    sub-16px contamination dark patches. Earlier stages have finer resolution:
      Stage 2 (40×40): captures ~2-cell-sized structures → contamination dark ✓
      Stage 3 (20×20): medium scale → scuffing patterns ✓
      Stage 4 (10×10): global bottle appearance → large defects ✓

    Each stage is GeM-pooled independently, then the three vectors are
    concatenated and projected via a small MLP to the final embedding.

    Parameters
    ----------
    stage_dims    : list of feature dimensions at each stage
                    ConvNeXt-Small: [192, 384, 768]  (stages 2, 3, 4)
    out_dim       : output embedding dimension for the head
    gem_p         : GeM pooling exponent
    """
    def __init__(
        self,
        stage_dims: List[int] = [192, 384, 768],
        out_dim:    int       = 512,
        gem_p:      float     = 3.0,
    ):
        super().__init__()
        total_dim = sum(stage_dims)

        # Separate learnable p per stage — coarser stages may benefit from higher p
        self.gems = nn.ModuleList([GeMPooling(p=gem_p) for _ in stage_dims])

        # Lightweight projection: concatenated → out_dim
        self.proj = nn.Sequential(
            nn.Linear(total_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
        )

    def forward(self, stage_features: List[torch.Tensor]) -> torch.Tensor:
        pooled = [gem(feat) for gem, feat in zip(self.gems, stage_features)]
        concat = torch.cat(pooled, dim=1)    # [B, sum(stage_dims)]
        return self.proj(concat)             # [B, out_dim]

In [12]:
class ChannelAttention(nn.Module):
    """
    Squeeze-and-Excitation style channel attention.
    Learns which of the C feature channels are most diagnostic for defects.
    Uses both average-pool and max-pool to capture different statistics.
    """
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, H, W]
        avg = x.mean(dim=[2, 3])                    # [B, C]
        mx  = x.flatten(2).max(dim=2).values        # [B, C]
        gate = torch.sigmoid(self.fc(avg) + self.fc(mx))
        return x * gate.unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    """
    Spatial attention — learns which [H, W] positions carry defect signal.

    For contamination dark:
      High activation at defect location → spatial gate amplifies it
      Low activation at clean glass → spatial gate suppresses it

    The result: defect regions have higher weight going into pooling,
    so their contribution to the pooled vector is proportionally larger.

    kernel_size=7: captures context in a 7×7 neighbourhood around each position.
    """
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        assert kernel_size % 2 == 1, 'kernel_size must be odd'
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size,
                              padding=kernel_size // 2, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_map = x.mean(dim=1, keepdim=True)           # [B, 1, H, W]
        max_map = x.max(dim=1, keepdim=True).values     # [B, 1, H, W]
        spatial_input = torch.cat([avg_map, max_map], dim=1)  # [B, 2, H, W]
        gate = torch.sigmoid(self.conv(spatial_input))  # [B, 1, H, W]
        return x * gate


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module.
    Applied after backbone, before pooling.

    Channel attention first (which features matter) → spatial attention
    (where they matter) → then pooling on the attended feature map.
    """
    def __init__(self, channels: int, reduction: int = 16, spatial_k: int = 7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(spatial_k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

In [13]:
class SpatialPyramidPooling(nn.Module):
    """
    Multi-scale pooling: pools feature map at 1×1, 2×2, and 4×4 grids,
    then concatenates. Expands the effective representation from C to
    C × (1 + 4 + 16) = 21C before projection.

    Why this helps over plain GAP:
      1×1 pool (1 cell):  global bottle appearance
      2×2 pool (4 cells): quadrant-level features (defect position matters)
      4×4 pool (16 cells): fine-grained spatial structure (local textures)

    A contamination dark patch in the lower-right quadrant affects the
    2×2 and 4×4 cells covering that region, without being diluted by the
    entire image as in GAP.
    """
    def __init__(self, levels: List[int] = [1, 2, 4]):
        super().__init__()
        self.levels = levels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, H, W]
        pooled = []
        for level in self.levels:
            pooled.append(
                F.adaptive_avg_pool2d(x, output_size=level)
                  .flatten(1)   # [B, C × level²]
            )
        return torch.cat(pooled, dim=1)   # [B, C × (1+4+16)]

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE
# ═══════════════════════════════════════════════════════════════════════════
# class SEBlock(nn.Module):
#     """
#     Squeeze-and-Excitation channel attention.

#     The head currently takes 768 features (ConvNeXt-Small GAP output) and
#     maps them to 1 logit. All 768 channels are treated equally.

    # SE attention adds a learned 'channel importance' weighting:
    # 1. Squeeze: global average across spatial dims (already done by GAP)
    # 2. Excitation: two FC layers learn a weight per channel in [0,1]
    # 3. Scale: multiply features by their learned channel weights

    # Channels associated with water-drop features get low weights.
    # Channels associated with contamination-dark get high weights.
    # The model learns this purely from the binary supervision signal.

    # reduction: how much to compress channels in the excitation bottleneck.
    #            16 means 768→48→768. Keeps parameter count tiny.
    # """
    # def __init__(self, channels: int, reduction: int = 16):
    #     super().__init__()
    #     mid = max(channels // reduction, 8)
    #     self.fc = nn.Sequential(
    #         nn.Linear(channels, mid, bias=False),
    #         nn.ReLU(inplace=True),
    #         nn.Linear(mid, channels, bias=False),
    #         nn.Sigmoid(),
    #     )

    # def forward(self, x: torch.Tensor) -> torch.Tensor:
    #     # x: [B, C]
    #     weights = self.fc(x)     # [B, C], values in (0, 1)
    #     return x * weights       # reweight each channel per sample




# Feature dimension after Global Average Pooling for each backbone
BACKBONE_FEAT_DIMS = {
    # Original
    'efficientnet_b0':                   1280,
    'mobilenet_v3_small':                 576,
    'resnet50':                          2048,
    # Recommended upgrades
    'convnext_small':                     768,   # ← try this first
    'convnext_base':                     1024,
    'tf_efficientnetv2_s':              1280,   # ← easiest drop-in swap
    'tf_efficientnetv2_m':              1280,
    'swin_tiny_patch4_window7_224':      768,   # ← transformer option
    'swin_small_patch4_window7_224':     768,
    'efficientnet_b2':                  1408,   # ← if staying with EffNet family
    'efficientnet_b3':                  1536,
    # 'hrnet_w32':                        480,
    'hrnet_w32': 2048,
    'hrnet_w40': 2048,
    'hrnet_w44': 2048,
    'hrnet_w48': 2048,
    'resnest50d':                       2048,
}


# def build_model(cfg: dict) -> nn.Module:
#     """
#     Construct and return the full BottleClassifier as an nn.Sequential-like model.

#     The model is a standard nn.Module wrapping:
#       1. A timm backbone (pretrained, head removed)
#       2. A custom binary classification head

#     timm's `num_classes=0` removes the original classification head and lets
#     `global_pool='avg'` handle Global Average Pooling, outputting [B, feat_dim].

#     Using timm (Py Torch Image Models) rather than torchvision gives access to
#     300+ pretrained architectures with a consistent API, including newer
#     EfficientNet variants trained on ImageNet-21k.
#     """
#     backbone_name = cfg['backbone']
    
#     data_cfg = timm.data.resolve_model_data_config(
#         timm.create_model(backbone_name, pretrained=False))
#     cfg['img_mean'] = list(data_cfg['mean'])
#     cfg['img_std']  = list(data_cfg['std'])

#     backbone = timm.create_model(
#         backbone_name, pretrained=True, num_classes=0, global_pool='avg'
#     )
#     feat_dim = BACKBONE_FEAT_DIMS.get(backbone_name, 768)

#     head = nn.Sequential(
    #     nn.Dropout(p=cfg['dropout1']),
    #     SEBlock(feat_dim, reduction=16),   # ← new: channel attention
    #     nn.Linear(feat_dim, cfg['hidden_dim']),
    #     nn.BatchNorm1d(cfg['hidden_dim']),
    #     nn.ReLU(inplace=True),
    #     nn.Dropout(p=cfg['dropout2']),
    #     nn.Linear(cfg['hidden_dim'], 1),
    # )

    # # Kaiming uniform initialisation for Linear layers (better than default)
    # for m in head.modules():
    #     if isinstance(m, nn.Linear):
    #         nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
    #         if m.bias is not None:
    #             nn.init.zeros_(m.bias)

    # # Combine backbone + head into a simple module
    # class BottleClassifier(nn.Module):
    #     def __init__(self, backbone, head):
    #         super().__init__()
    #         self.backbone = backbone
    #         self.head     = head

    #     def forward(self, x):
    #         return self.head(self.backbone(x))  # [B,1] logits

    # model = BottleClassifier(backbone, head)
    # log.info(
    #     'Model: %s | total params: %s | trainable: %s',
    #     backbone_name,
    #     f'{sum(p.numel() for p in model.parameters()):,}',
    #     f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}',
    # )
    # return model

def build_model(cfg: dict) -> nn.Module:
    backbone_name = cfg['backbone']
    data_cfg = timm.data.resolve_model_data_config(
        timm.create_model(backbone_name, pretrained=False))
    cfg['img_mean'] = list(data_cfg['mean'])
    cfg['img_std']  = list(data_cfg['std'])

    is_hrnet = backbone_name.startswith('hrnet')
    use_msff = cfg.get('use_msff', True)
    use_cbam = cfg.get('use_cbam', True)
    gem_p    = cfg.get('gem_p',   3.0)
    embed    = cfg.get('hidden_dim', 512)

    if is_hrnet and use_msff:
        return _build_hrnet_multibranch(cfg, backbone_name,
                                         use_cbam, gem_p, embed)
    elif use_msff and not is_hrnet:
        return _build_convnext_msff(cfg, backbone_name,
                                     use_cbam, gem_p, embed)
    else:
        return _build_standard(cfg, backbone_name,
                                use_cbam, gem_p, embed)


def _build_hrnet_multibranch(cfg, backbone_name, use_cbam, gem_p, embed):
    """
    HRNet multi-branch fusion — the ideal path for small defect detection.

    HRNet-W32 after stage4 produces 4 parallel feature maps:
      Branch 1: [B, 32,  H/4,  W/4]   ← highest resolution (80×80 at 320px)
      Branch 2: [B, 64,  H/8,  W/8]
      Branch 3: [B, 128, H/16, W/16]
      Branch 4: [B, 256, H/32, W/32]

    We bypass the standard incre_modules (which just increase channels for
    ImageNet classification) and instead apply GeM directly to each branch.
    This preserves the fine spatial structure that incre_modules would dilute.

    Why not use the standard 2048-feature path?
    The incre_modules + GAP are designed for 1000-class ImageNet.
    For binary defect classification, the branch 1 features (80×80 at 320px)
    contain the contamination dark signal that GAP would average away.
    """
    # Load backbone WITHOUT the final classification modules
    # We extract raw stage4 outputs via a hook
    backbone = timm.create_model(
        backbone_name, pretrained=True, num_classes=0, global_pool=''
    )

    # HRNet-W32 branch channel dimensions after stage4
    hrnet_branch_dims = {
        'hrnet_w32': [32,  64,  128, 256],
        'hrnet_w40': [40,  80,  160, 320],
        'hrnet_w44': [44,  88,  176, 352],
        'hrnet_w48': [48,  96,  192, 384],
    }
    branch_dims = hrnet_branch_dims.get(backbone_name, [32, 64, 128, 256])
    total_branch_dim = sum(branch_dims)

    # CBAM per branch (optional)
    cbam_modules = nn.ModuleList([
        CBAM(d, reduction=8, spatial_k=7) if use_cbam else nn.Identity()
        for d in branch_dims
    ])

    # GeM pooling per branch
    gem_modules = nn.ModuleList([GeMPooling(p=gem_p) for _ in branch_dims])

    # Projection from concatenated branches to embedding
    fusion = nn.Sequential(
        nn.Linear(total_branch_dim, embed),
        nn.BatchNorm1d(embed),
        nn.GELU(),
        nn.Dropout(p=cfg.get('dropout1', 0.4)),
    )

    head = nn.Sequential(
        nn.Linear(embed, cfg.get('head_hidden', 256)),
        nn.BatchNorm1d(cfg.get('head_hidden', 256)),
        nn.ReLU(inplace=True),
        nn.Dropout(p=cfg.get('dropout2', 0.2)),
        nn.Linear(cfg.get('head_hidden', 256), 1),
    )
    for m in head.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    class HRNetBottleClassifier(nn.Module):
        def __init__(self, backbone, cbams, gems, fusion, head):
            super().__init__()
            self.backbone = backbone
            self.cbams    = cbams
            self.gems     = gems
            self.fusion   = fusion
            self.head     = head
            self._branches: List[torch.Tensor] = []

            # Hook to capture the 4 parallel branches from stage4 output
            # before HRNet's incre_modules process them
            self._register_hrnet_hook()

        def _register_hrnet_hook(self):
            """
            HRNet stage4 outputs a list of 4 tensors (one per branch).
            We register a forward hook on stage4 to capture them before
            the incre_modules collapse them into a single 2048-channel map.
            """
            def _hook(module, input, output):
                # output is a list [branch1, branch2, branch3, branch4]
                self._branches = output if isinstance(output, (list, tuple)) \
                                 else [output]

            self.backbone.stage4.register_forward_hook(_hook)

        def forward(self, x):
            # Run full forward pass to trigger the hook
            self.backbone(x)

            # self._branches now contains the 4 stage4 outputs
            branches = self._branches

            # Apply CBAM and GeM to each branch
            pooled = []
            for i, (branch, cbam, gem) in enumerate(
                zip(branches, self.cbams, self.gems)
            ):
                attended = cbam(branch)   # spatial + channel attention
                pooled.append(gem(attended))  # [B, C_i]

            concat = torch.cat(pooled, dim=1)  # [B, sum(branch_dims)]
            embed  = self.fusion(concat)        # [B, embed]
            return self.head(embed)             # [B, 1]

    model = HRNetBottleClassifier(backbone, cbam_modules, gem_modules, fusion, head)
    total = sum(p.numel() for p in model.parameters())
    log.info('Model: %s multi-branch + CBAM=%s + GeM(p=%.1f) | params: %s',
             backbone_name, use_cbam, gem_p, f'{total:,}')
    return model


# Add to CFG:
CFG['use_msff']    = True    # multi-scale feature fusion
CFG['use_cbam']    = True    # spatial + channel attention
CFG['gem_p']       = 3.0    # GeM pooling exponent (learnable)
CFG['head_hidden'] = 256     # intermediate head dimension


def freeze_backbone(model: nn.Module) -> None:
    """
    Freeze all backbone parameters.
    For HRNet multi-branch model, also ensures fusion/cbam/gem stay trainable
    since they are part of our custom head, not the pretrained backbone.
    """
    for p in model.backbone.parameters():
        p.requires_grad = False

    # These are our custom modules — always trainable
    for attr in ['fusion', 'cbams', 'gems', 'msff', 'head']:
        module = getattr(model, attr, None)
        if module is not None:
            for p in module.parameters():
                p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info('Backbone frozen. Custom head trainable params: %s', f'{trainable:,}')


def unfreeze_backbone_last_n(model: nn.Module, n: Optional[int] = 3) -> None:
    if n is None:
        for p in model.backbone.parameters():
            p.requires_grad = True
        log.info('Full backbone unfrozen.')

    elif hasattr(model.backbone, 'stage4'):
        # ── HRNet family ──────────────────────────────────────────────────
        # "Last N" maps to the final N major components of HRNet.
        # The most impactful layers to unfreeze are:
        #   stage4 (the 4-branch fusion stage — most task-specific)
        #   transition3 (creates the 4th branch feeding into stage4)
        #   stage3 (3-branch stage)
        #   incre_modules + final_layer (classification modules)
        #
        # n=1: stage4 only
        # n=2: transition3 + stage4
        # n=3: stage3 + transition3 + stage4  ← default, recommended
        # n=4: stage2 + all above
        # None: everything

        hrnet_layers = [
            'stage2', 'transition2',
            'stage3', 'transition3',
            'stage4',
            'incre_modules', 'downsamp_modules', 'final_layer',
        ]
        # Unfreeze the last n of these (from the end of the list)
        layers_to_unfreeze = hrnet_layers[max(0, len(hrnet_layers) - n):]

        # First freeze everything
        for p in model.backbone.parameters():
            p.requires_grad = False

        # Then unfreeze the target layers
        for layer_name in layers_to_unfreeze:
            layer = getattr(model.backbone, layer_name, None)
            if layer is not None:
                for p in layer.parameters():
                    p.requires_grad = True

        log.info('HRNet unfreeze: %s', layers_to_unfreeze)

    elif hasattr(model.backbone, 'stages'):
        # ConvNeXt family (existing code)
        stages = list(model.backbone.stages.children())
        total  = len(stages)
        for i, s in enumerate(stages):
            for p in s.parameters():
                p.requires_grad = (i >= total - n)
        if hasattr(model.backbone, 'norm_pre'):
            for p in model.backbone.norm_pre.parameters():
                p.requires_grad = True

    elif hasattr(model.backbone, 'blocks'):
        # EfficientNet family (existing code)
        blocks = list(model.backbone.blocks.children())
        total  = len(blocks)
        for i, b in enumerate(blocks):
            for p in b.parameters():
                p.requires_grad = (i >= total - n)
        for ln in ['conv_head', 'bn2', 'act2']:
            layer = getattr(model.backbone, ln, None)
            if layer:
                for p in layer.parameters():
                    p.requires_grad = True

    else:
        children = list(model.backbone.children())
        total    = len(children)
        for i, c in enumerate(children):
            for p in c.parameters():
                p.requires_grad = (i >= total - n)

    # Head always trainable
    for p in model.head.parameters():
        p.requires_grad = True
    if hasattr(model, 'fusion'):
        for p in model.fusion.parameters():
            p.requires_grad = True
    if hasattr(model, 'cbams'):
        for p in model.cbams.parameters():
            p.requires_grad = True
    if hasattr(model, 'gems'):
        for p in model.gems.parameters():
            p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info('Trainable params after unfreeze: %s', f'{trainable:,}')


print('Model architecture functions ready.')

Model architecture functions ready.


---
## 7 — Focal Loss

### What it does
A modified cross-entropy loss that automatically **reduces the weight of easy examples** and **increases the weight of hard ones**.

### The formula
```
FL(p_t) = -α_t · (1 - p_t)^γ · log(p_t)
```
- `p_t` = probability the model assigned to the correct class
- `(1 - p_t)^γ` = **focusing factor**: if the model is confident (p_t ≈ 1), this factor ≈ 0 → small loss. If the model is uncertain (p_t ≈ 0.5), this factor ≈ 0.25 → large loss.
- `α_t` = class-specific weight. Set higher for FAULTY to penalise missed defects more.

### Why not standard BCE?
With standard BCE, on a balanced batch the model can achieve a reasonable loss by correctly classifying the obvious easy cases (clear glass shard, perfectly clean bottle) and ignoring the hard borderline cases (contamination_light near the 180px threshold). Focal loss forces the model to keep learning from the hard cases even when easy ones are already well-classified.

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════════════════════
def focal_loss(
    logits:         torch.Tensor,
    targets:        torch.Tensor,
    gamma:          float = 2.0,
    alpha:          float = 0.35,
    sample_weights: Optional[torch.Tensor] = None,
    reduction:      str   = 'mean',
) -> torch.Tensor:
    """
    Binary Focal Loss with optional per-sample weighting.

    When sample_weights is provided (training), hard examples receive a
    higher loss contribution proportional to their weight value.
    When sample_weights is None (validation/test), standard focal loss runs.

    Parameters
    ----------
    logits         : raw model output [B] or [B,1] — before sigmoid
    targets        : binary float targets {0.0, 1.0} [B] or [B,1]
    gamma          : focusing parameter (0 = standard BCE, 2 = default)
    alpha          : FAULTY class weight
    sample_weights : per-sample multipliers [B]; None = all weights = 1.0
    reduction      : 'mean' | 'sum' | 'none'
    """
    logits  = logits.view(-1)
    targets = targets.view(-1)

    bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    probs   = torch.sigmoid(logits)
    p_t     = probs * targets + (1.0 - probs) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss    = alpha_t * (1.0 - p_t) ** gamma * bce

    # Apply per-sample weights when provided
    if sample_weights is not None:
        weights = sample_weights.view(-1).to(logits.device)
        loss    = loss * weights

    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    return loss


print('Focal loss function ready.')

Focal loss function ready.


---
## 8 — Learning Rate Schedule

### What it does
Provides a two-phase schedule that matches the two-phase training strategy:
- **Phase 1** (epochs 0–7): Head warmup at constant high LR
- **Phase 2** (epochs 8–49): Fine-tuning starting from a lower LR, with cosine annealing that decays smoothly to near-zero

### Why cosine annealing?
Cosine annealing decays the LR smoothly (following the upper half of a cosine curve) rather than abruptly. This prevents the model from 'bouncing' around a minimum at the end of training. The cosine shape has become the standard for fine-tuning pretrained networks.

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# LEARNING RATE SCHEDULE
# ═══════════════════════════════════════════════════════════════════════════

def lr_lambda(epoch: int, cfg: dict) -> float:
    """
    Piecewise LR multiplier used with torch.optim.lr_scheduler.LambdaLR.

    The scheduler calls this function every epoch and multiplies the base LR
    (lr_phase1) by the returned float.

    Segments
    ────────
    0 … warmup_epochs:          Linear ramp 0 → 1.0  (prevents gradient explosion at start)
    warmup … phase1_end:        Constant 1.0          (stable head training)
    phase1_end … total_epochs:  Cosine decay from (lr_phase2/lr_phase1) → (min_lr/lr_phase1)

    Note: LambdaLR multiplies the *base LR* (lr_phase1) by this value, so
    phase-2 must return lr_phase2/lr_phase1 at its start.
    """
    w  = cfg['warmup_epochs']
    p1 = cfg['phase1_epochs']
    T  = cfg['total_epochs']
    lr1     = cfg['lr_phase1']
    lr2     = cfg['lr_phase2']
    min_lr  = cfg['min_lr']

    if epoch < w:
        # Linear warmup: epoch 0 → near-zero, epoch w → lr_phase1
        return (epoch + 1) / max(w, 1)

    if epoch < p1:
        # Constant phase-1 LR
        return 1.0

    # Cosine annealing for phase 2
    # Progress goes from 0 (epoch=p1) to 1 (epoch=T)
    p2_length = max(T - p1, 1)
    progress  = (epoch - p1) / p2_length
    cosine    = 0.5 * (1.0 + math.cos(math.pi * progress))
    # Interpolate between lr2 and min_lr
    target_lr = min_lr + (lr2 - min_lr) * cosine
    # Return as a multiplier of the base LR (lr_phase1)
    return target_lr / lr1


def build_optimizer_and_scheduler(model: nn.Module, cfg: dict):
    """
    Build AdamW optimizer and LambdaLR scheduler.

    Why AdamW?
    ──────────
    AdamW decouples weight decay from the gradient update (unlike vanilla
    Adam which conflates them). This gives cleaner regularisation and is
    now the standard for fine-tuning transformers and CNNs alike.
    """
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['lr_phase1'],
        weight_decay=cfg['weight_decay'],
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: lr_lambda(epoch, cfg),
    )

    return optimizer, scheduler


print('LR schedule functions ready.')

LR schedule functions ready.


---
## 9 — Training Engine

### What it does
- `train_one_epoch`: one full pass over the training DataLoader, accumulating loss and updating weights
- `evaluate`: one pass over a DataLoader with no gradients, collecting predictions
- `train`: orchestrates the full training loop with phase switching, early stopping, and checkpointing

### AMP (Automatic Mixed Precision)
Training with fp16 (half precision) instead of fp32 for the forward pass and loss:
- ~2× faster on modern NVIDIA GPUs (tensor cores are optimised for fp16)
- ~2× less GPU memory (larger batch sizes)
- The `GradScaler` prevents fp16 underflow by scaling the loss before backprop and unscaling the gradients before the weight update

In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING ENGINE
# ═══════════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, scaler, cfg, device,
                    apply_sample_weights=True):
    model.train()
    running_loss = 0.0
    accum        = cfg.get('accumulate_grad_batches', 1)
    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels, weights) in enumerate(loader):
        images  = images.to(device,  non_blocking=True)
        labels  = labels.to(device,  non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        effective_weights = weights if apply_sample_weights \
                            else torch.ones_like(weights)

        with torch.amp.autocast(device_type=device.type,
                                enabled=cfg['amp'] and device.type == 'cuda'):
            logits = model(images).squeeze(1)
            loss   = focal_loss(logits, labels, cfg['focal_gamma'],
                                cfg['focal_alpha'],
                                sample_weights=effective_weights) / accum

        scaler.scale(loss).backward()

        if (step + 1) % accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum

    return running_loss / max(len(loader), 1)

 


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    cfg: dict,
    threshold: float = 0.5,
) -> Dict[str, float]:
    """
    Run inference on a DataLoader and return all metrics.

    @torch.no_grad() disables gradient tracking, reducing memory and
    speeding up inference by ~20%.

    Returns
    -------
    dict with keys: loss, f1, precision, recall, accuracy, roc_auc,
                    pr_auc, probs (numpy array), labels (numpy array)
    """
    model.eval()
    all_probs:  List[float] = []
    all_labels: List[int]   = []
    total_loss = 0.0

    for batch in loader:
        # Unpack 3 elements — weight ignored during evaluation
        images, labels, _weights = batch
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=cfg['amp'] and device.type == 'cuda'
        ):
            logits = model(images).squeeze(1)
            loss   = focal_loss(
                logits, labels,
                gamma=cfg['focal_gamma'],
                alpha=cfg['focal_alpha'],
                sample_weights=None,   # no weighting during eval
            )

        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        total_loss += loss.item()

    probs_arr  = np.array(all_probs,  dtype=np.float32)
    labels_arr = np.array(all_labels, dtype=np.int32)
    preds_arr  = (probs_arr >= threshold).astype(int)

    try:
        roc_auc = roc_auc_score(labels_arr, probs_arr)
        pr_auc  = average_precision_score(labels_arr, probs_arr)
    except Exception:
        roc_auc = pr_auc = 0.0

    return {
        'loss':      total_loss / max(len(loader), 1),
        'f1':        f1_score(labels_arr, preds_arr, zero_division=0),
        'precision': precision_score(labels_arr, preds_arr, zero_division=0),
        'recall':    recall_score(labels_arr, preds_arr, zero_division=0),
        'accuracy':  accuracy_score(labels_arr, preds_arr),
        'roc_auc':   roc_auc,
        'pr_auc':    pr_auc,
        'probs':     probs_arr,
        'labels':    labels_arr,
    }
    


def build_phase1_optimizer(model, cfg):
    """
    Phase 1: only head parameters are trainable.
    High LR — head weights are random and need fast correction.
    """
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['lr_head'],
        weight_decay=cfg['weight_decay'],
        betas=(0.9, 0.999),
    )


def build_phase2_optimizer(model, cfg):
    """
    Phase 2: unfrozen backbone layers + full custom head.
    For HRNet, the backbone LR is lower since stage3/4 are already
    somewhat task-relevant from ImageNet pretraining.
    """
    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]

    # Collect all non-backbone trainable params (fusion, cbams, gems, head)
    head_params = []
    for attr in ['fusion', 'cbams', 'gems', 'msff', 'head']:
        module = getattr(model, attr, None)
        if module is not None:
            head_params.extend(list(module.parameters()))

    return torch.optim.AdamW([
        {'params': bb_params,   'lr': cfg['lr_phase2']},
        {'params': head_params, 'lr': cfg['lr_phase2'] * 5},
    ], weight_decay=cfg['weight_decay'])


def build_phase3_optimizer(model, cfg):
    """
    Phase 3: full unfreeze with differential LR.
    Backbone at 5e-6 — gentle enough to reshape features without
    destroying ImageNet weights. Head at 3e-5.
    """
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.head.parameters())

    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': cfg['lr_phase3_bb']},
        {'params': head_params,     'lr': cfg['lr_phase3_hd']},
    ],
        weight_decay=cfg['weight_decay'],
    )


def build_phase4_optimizer(model, cfg):
    """
    Phase 4: cooldown pass — everything at minimum LR.
    Settles the model into a clean local minimum.
    Fine-tunes subtle boundary decisions without destabilising.
    """
    return torch.optim.AdamW(
        model.parameters(),
        lr=cfg['lr_phase4'],
        weight_decay=cfg['weight_decay'] * 0.1,  # reduce regularisation
    )


def build_cosine_scheduler(optimizer, n_epochs, warmup=0, min_lr=1e-7):
    """
    Cosine annealing with optional linear warmup.
    Works with any optimizer regardless of number of param groups.

    warmup : number of epochs for linear ramp-up
    """
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / max(warmup, 1)
        progress = (epoch - warmup) / max(n_epochs - warmup, 1)
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        # min_lr as fraction of base LR
        base_lr  = optimizer.param_groups[0]['initial_lr'] \
                   if 'initial_lr' in optimizer.param_groups[0] \
                   else optimizer.param_groups[0]['lr']
        return max(min_lr / base_lr, cosine)

    # Store initial LR for each param group (needed by lambda above)
    for pg in optimizer.param_groups:
        pg['initial_lr'] = pg['lr']

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)



def train(model, train_loader, val_loader, cfg, device):
    """
    Four-phase training loop.

    Only change vs previous: passes apply_sample_weights=(current_phase >= 2)
    to train_one_epoch. Weights are OFF in phase 1, ON from phase 2 onwards.
    """
    model = model.to(device)
    os.makedirs(Path(cfg['checkpoint_path']).parent, exist_ok=True)
    os.makedirs(cfg['output_dir'], exist_ok=True)

    p1_end = cfg['phase1_epochs']
    p2_end = p1_end + cfg['phase2_epochs']
    p3_end = p2_end + cfg['phase3_epochs']

    scaler = torch.amp.GradScaler(
        device=device.type,
        enabled=cfg['amp'] and device.type == 'cuda',
    )

    freeze_backbone(model)
    optimizer = build_phase1_optimizer(model, cfg)
    scheduler = build_cosine_scheduler(
        optimizer, n_epochs=cfg['phase1_epochs'],
        warmup=cfg['warmup_epochs'], min_lr=cfg['min_lr'],
    )
    current_phase = 1
    log.info('Phase 1 — backbone frozen | sample weights DISABLED '
             '(head is random; enabled from phase 2).')

    best_f1    = 0.0
    no_improve = 0
    history    = {
        'train_loss': [], 'val_loss': [],
        'val_f1': [], 'val_recall': [],
        'lr_backbone': [], 'lr_head': [], 'phase': [],
    }

    for epoch in range(cfg['total_epochs']):

        # ── Phase transitions ─────────────────────────────────────────────
        if epoch == p1_end and current_phase == 1:
            log.info('=== Phase 2: unfreezing last 3 blocks | weights ENABLED ===')
            unfreeze_backbone_last_n(model, n=3)
            optimizer = build_phase2_optimizer(model, cfg)
            scheduler = build_cosine_scheduler(
                optimizer, n_epochs=cfg['phase2_epochs'],
                warmup=2, min_lr=cfg['min_lr'],
            )
            current_phase = 2

        elif epoch == p2_end and current_phase == 2:
            log.info('=== Phase 3: full backbone unfreeze ===')
            unfreeze_backbone_last_n(model, n=None)
            optimizer = build_phase3_optimizer(model, cfg)
            scheduler = build_cosine_scheduler(
                optimizer, n_epochs=cfg['phase3_epochs'],
                warmup=3, min_lr=cfg['min_lr'],
            )
            current_phase = 3

        elif epoch == p3_end and current_phase == 3:
            log.info('=== Phase 4: cooldown ===')
            optimizer = build_phase4_optimizer(model, cfg)
            scheduler = build_cosine_scheduler(
                optimizer, n_epochs=cfg['phase4_epochs'],
                warmup=0, min_lr=cfg['min_lr'],
            )
            current_phase = 4

        # ── Train — weights OFF in phase 1, ON from phase 2 ──────────────
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler, cfg, device,
            apply_sample_weights=(current_phase >= 2),   # ← the key change
        )
        scheduler.step()

        val_m = evaluate(model, val_loader, device, cfg,
                         threshold=cfg['default_threshold'])

        lrs         = [pg['lr'] for pg in optimizer.param_groups]
        lr_backbone = lrs[0]
        lr_head     = lrs[-1]

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_m['loss'])
        history['val_f1'].append(val_m['f1'])
        history['val_recall'].append(val_m['recall'])
        history['lr_backbone'].append(lr_backbone)
        history['lr_head'].append(lr_head)
        history['phase'].append(current_phase)

        log.info(
            'Ep %3d [ph%d | w=%s] | loss %.4f→%.4f | F1 %.4f | '
            'rec %.4f | lr_bb %.1e lr_hd %.1e',
            epoch + 1, current_phase,
            'ON' if current_phase >= 2 else 'OFF',
            train_loss, val_m['loss'], val_m['f1'], val_m['recall'],
            lr_backbone, lr_head,
        )

        if val_m['f1'] > best_f1 + cfg['early_stop_min_delta']:
            best_f1    = val_m['f1']
            no_improve = 0
            torch.save({
                'epoch':      epoch + 1,
                'state_dict': model.state_dict(),
                'val_f1':     best_f1,
                'phase':      current_phase,
                'cfg':        cfg,
            }, cfg['checkpoint_path'])
            log.info('  ↑ New best val_f1=%.4f saved.', best_f1)
        else:
            no_improve += 1

        if current_phase >= 3 and no_improve >= cfg['early_stop_patience']:
            log.info('Early stopping at epoch %d.', epoch + 1)
            break

    log.info('Training complete. Best val_f1=%.4f', best_f1)
    return history


print('Training engine ready.')

Training engine ready.


---
## 10 — Hyperparameter Tuning (Optuna)

### What it does
Runs `n_trials` independent training runs with different hyperparameter combinations, guided by the **Tree-structured Parzen Estimator (TPE)** algorithm. TPE is a Bayesian optimisation method that builds a probabilistic model of which hyperparameters produce good results, and samples more from promising regions.

### Safety constraint
Any trial where `val_recall_faulty < 0.99` scores 0.0. This prevents Optuna from finding configurations that achieve high F1 by sacrificing recall (i.e. passing faulty bottles in exchange for fewer false rejections). In industrial quality control, a passed faulty bottle is far more costly than a rejected good bottle.

### When to run this
Run Optuna **before** full training with a short budget per trial (12–15 epochs). Then update CFG with the best parameters and run the full training (Stage 9).

In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# HYPERPARAMETER TUNING
# ═══════════════════════════════════════════════════════════════════════════

def optuna_objective(
    trial: optuna.Trial,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    base_cfg: dict,
    n_epochs: int = 12,
) -> float:
    """
    Single Optuna trial — 2-phase training on a short budget.

    Why only 2 phases for search?
    ─────────────────────────────
    12 epochs is not enough compute to meaningfully test phases 3 and 4.
    The parameters that most influence final F1 — focal_alpha, focal_gamma,
    dropout, and learning rates — are all expressed within the first 2 phases.
    Optuna finds the best values here; you run the full 4-phase training once
    with those best values applied to CFG.
    """
    cfg = copy.deepcopy(base_cfg)

    # ── Sample hyperparameters ────────────────────────────────────────────
    # lr_head: phase 1 LR (head only, backbone frozen)
    cfg['lr_head']      = trial.suggest_float('lr_head',      1e-4,  5e-3, log=True)
    # lr_phase2: phase 2 backbone LR (last 3 blocks unfrozen)
    cfg['lr_phase2']    = trial.suggest_float('lr_phase2',    1e-6,  5e-4, log=True)
    cfg['focal_gamma']  = trial.suggest_float('focal_gamma',  1.0,   4.0)
    cfg['focal_alpha']  = trial.suggest_float('focal_alpha',  0.3,   0.6)  # tightened around 0.42
    cfg['dropout1']     = trial.suggest_float('dropout1',     0.2,   0.6)
    cfg['dropout2']     = trial.suggest_float('dropout2',     0.1,   0.4)
    cfg['hidden_dim']   = trial.suggest_categorical('hidden_dim', [128, 256, 512])
    cfg['weight_decay'] = trial.suggest_float('weight_decay', 1e-5,  1e-3, log=True)
    cfg['batch_size']   = trial.suggest_categorical('batch_size', [32, 64, 128])

    # ── Set phase epoch boundaries for short 2-phase budget ──────────────
    p1 = max(2, int(n_epochs * 0.20))   # 20% for head warmup (≈2 epochs at n=12)
    p2 = n_epochs - p1                  # rest for partial backbone fine-tune
    cfg['total_epochs']   = n_epochs
    cfg['phase1_epochs']  = p1
    cfg['phase2_epochs']  = p2
    cfg['phase3_epochs']  = 0           # not used in short search
    cfg['phase4_epochs']  = 0           # not used in short search
    cfg['warmup_epochs']  = min(2, p1)  # at most 2 warmup epochs
    cfg['early_stop_patience'] = 9999   # disable early stopping during search

    # ── Build model ───────────────────────────────────────────────────────
    model = build_model(cfg).to(DEVICE)
    freeze_backbone(model)

    # Use the same phase-specific optimizer builders as the full train() loop
    optimizer = build_phase1_optimizer(model, cfg)
    scheduler = build_cosine_scheduler(
        optimizer,
        n_epochs=p1,
        warmup=cfg['warmup_epochs'],
        min_lr=cfg['min_lr'],
    )
    scaler = torch.amp.GradScaler(
        device='cuda',
        enabled=cfg['amp'] and DEVICE.type == 'cuda'
    )

    current_phase = 1
    best_f1 = 0.0

    for epoch in range(n_epochs):

        # ── Phase 1 → Phase 2 switch ──────────────────────────────────────
        if epoch == p1 and current_phase == 1:
            unfreeze_backbone_last_n(model, n=3)
            optimizer = build_phase2_optimizer(model, cfg)
            scheduler = build_cosine_scheduler(
                optimizer,
                n_epochs=p2,
                warmup=1,           # 1-epoch warmup at phase switch to avoid LR spike
                min_lr=cfg['min_lr'],
            )
            current_phase = 2

        # ── Train + validate ──────────────────────────────────────────────
        train_one_epoch(model, train_loader, optimizer, scaler, cfg, DEVICE)
        scheduler.step()

        val_m  = evaluate(model, val_loader, DEVICE, cfg)
        f1_val = val_m['f1']
        recall = val_m['recall']

        # Report to Optuna for pruning decisions
        trial.report(f1_val, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        best_f1 = max(best_f1, f1_val)

    # ── Safety constraint ─────────────────────────────────────────────────
    # Any config that misses too many faulty bottles scores zero
    # Prevents Optuna from trading safety for precision
    # if recall < cfg['min_recall_faulty']:
    #     return 0.0

    return best_f1
   


def run_hyperparameter_search(
    train_loader: DataLoader,
    val_loader:   DataLoader,
    cfg: dict,
) -> dict:
    """
    Run the full Optuna hyperparameter search and return the best config.

    TPE sampler: builds a Gaussian mixture model of good vs bad parameter
    regions, then samples from the 'good' model (Expected Improvement).

    Median pruner: kills trials whose intermediate F1 is below the median
    of completed trials at the same epoch, avoiding wasting compute.
    """
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )

    study.optimize(
        lambda trial: optuna_objective(
            trial, train_loader, val_loader, cfg, n_epochs=cfg['optuna_n_epochs']
        ),
        n_trials=cfg['optuna_n_trials'],
        show_progress_bar=True,
    )

    best = study.best_trial
    log.info('Best Optuna trial #%d — val_f1=%.4f', best.number, best.value)
    for k, v in best.params.items():
        log.info('  %-20s = %s', k, v)

    # Build optimised config by patching best params into base config
    best_cfg = copy.deepcopy(cfg)
    for k, v in best.params.items():
        if k in best_cfg:
            best_cfg[k] = v

    return best_cfg, study


print('Hyperparameter tuning functions ready.')

Hyperparameter tuning functions ready.


---
## 11 — Threshold Calibration

### What it does
After training, the model outputs `P(FAULTY)` in [0, 1]. The default threshold of 0.5 is almost never optimal. This function sweeps all candidate thresholds from the precision-recall curve and selects the one that:
1. **Must satisfy**: `recall_faulty ≥ 0.99` (safety gate — never compromise this)
2. **Then maximises**: F1 score

### Why calibrate on val, not test?
The threshold is a hyperparameter. If we calibrate on the test set, we are effectively training on it, which would give an optimistic and misleading final F1. Calibrating on the validation set and reporting test performance is the correct protocol.

### Innovation
The safety-constrained threshold search is non-standard. Most practitioners just pick the F1-maximising threshold. Adding the `min_recall_faulty` constraint formalises the industrial requirement that **missing a faulty bottle is worse than falsely rejecting a good one**.

In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# THRESHOLD CALIBRATION
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics_at_threshold(
    labels: np.ndarray,
    probs:  np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    """
    Compute all binary classification metrics given a specific threshold.

    This is the single source of truth for metric computation.
    All evaluation functions call this rather than implementing metrics inline.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    prec_faulty = tp / max(tp + fp, 1)
    rec_faulty  = tp / max(tp + fn, 1)
    f1_faulty   = 2 * prec_faulty * rec_faulty / max(prec_faulty + rec_faulty, 1e-9)

    try:
        roc = roc_auc_score(labels, probs)
        pr  = average_precision_score(labels, probs)
    except Exception:
        roc = pr = 0.0

    return {
        'threshold':         threshold,
        'f1_faulty':         f1_faulty,
        'f1_macro':          f1_score(labels, preds, average='macro', zero_division=0),
        'precision_faulty':  prec_faulty,
        'recall_faulty':     rec_faulty,
        'accuracy':          accuracy_score(labels, preds),
        'roc_auc':           roc,
        'pr_auc':            pr,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

# compute_metrics_at_threshold is UNCHANGED — keep as-is.


def calibrate_threshold(
    val_probs,
    val_labels,
    min_recall=0.99,
):
    """
    Find the optimal decision threshold on the validation set.

    Two-pass search (new vs single PR-curve pass before)
    ─────────────────────────────────────────────────────
    Pass 1 — PR curve sweep:
        sklearn's precision_recall_curve gives one threshold per unique
        prediction score. Fast, but misses the optimum when score density
        is uneven (previous run found τ=0.117 when τ=0.130 gave +0.013 F1).

    Pass 2 — Fine-grained sweep at 0.002 step [0.05, 0.70]:
        Catches whatever Pass 1 missed. Runs ~325 threshold evaluations —
        negligible cost vs training time.

    Safety gate: any τ with recall_faulty < min_recall is skipped.
    If no τ passes the gate, the unconstrained best is returned with a warning.
    """
    best_f1        = 0.0
    best_tau       = 0.5
    fallback_f1    = 0.0
    fallback_tau   = 0.5

    # ── Pass 1: PR curve sweep ─────────────────────────────────────────────
    precision_arr, recall_arr, thresholds = precision_recall_curve(
        val_labels, val_probs
    )
    thresholds = np.append(thresholds, 1.0)

    for tau, _prec, rec in zip(thresholds, precision_arr, recall_arr):
        preds  = (val_probs >= tau).astype(int)
        f1_val = f1_score(val_labels, preds, zero_division=0)
        if f1_val > fallback_f1:
            fallback_f1  = f1_val
            fallback_tau = float(tau)
        if rec < min_recall:
            continue
        if f1_val > best_f1:
            best_f1  = f1_val
            best_tau = float(tau)

    # ── Pass 2: fine-grained sweep at 0.002 step ───────────────────────────
    # Catches thresholds the PR curve might space too coarsely.
    for tau in np.arange(0.05, 0.70, 0.002):
        preds  = (val_probs >= tau).astype(int)
        rec    = recall_score(val_labels, preds, zero_division=0)
        f1_val = f1_score(val_labels, preds, zero_division=0)
        if f1_val > fallback_f1:
            fallback_f1  = f1_val
            fallback_tau = float(tau)
        if rec < min_recall:
            continue
        if f1_val > best_f1:
            best_f1  = f1_val
            best_tau = float(tau)

    if best_f1 == 0.0:
        log.warning(
            'No threshold satisfies recall >= %.2f. Using unconstrained best (τ=%.4f).',
            min_recall, fallback_tau,
        )
        best_tau = fallback_tau

    best_metrics = compute_metrics_at_threshold(val_labels, val_probs, best_tau)
    log.info(
        'Calibrated threshold=%.4f | F1(FAULTY)=%.4f | '
        'Recall=%.4f | Prec=%.4f',
        best_tau, best_metrics['f1_faulty'],
        best_metrics['recall_faulty'], best_metrics['precision_faulty'],
    )
    return best_tau, best_metrics


print('Threshold calibration ready.')

Threshold calibration ready.


---
## 12 — Evaluation & Visualisation

In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION & PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def print_evaluation_report(metrics: dict, split: str = 'Test') -> None:
    """Human-readable evaluation summary printed to stdout."""
    sep = '─' * 62
    target_f1 = 0.98
    status = '✓ PASSED' if metrics['f1_faulty'] >= target_f1 else '✗ BELOW TARGET'
    print(f'\n{sep}')
    print(f'  BOTTLE INSPECTION — {split.upper()} SET EVALUATION')
    print(f'{sep}')
    print(f'  Decision threshold   : {metrics["threshold"]:.4f}')
    print(f'{sep}')
    print(f'  F1 (FAULTY class)    : {metrics["f1_faulty"]:.4f}   ← primary KPI')
    print(f'  Recall    (FAULTY)   : {metrics["recall_faulty"]:.4f}   ← safety metric')
    print(f'  Precision (FAULTY)   : {metrics["precision_faulty"]:.4f}')
    print(f'  F1 (macro)           : {metrics["f1_macro"]:.4f}')
    print(f'  Accuracy             : {metrics["accuracy"]:.4f}')
    print(f'  ROC-AUC              : {metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC               : {metrics["pr_auc"]:.4f}')
    print(f'{sep}')
    print(f'  Confusion matrix:')
    print(f'    TP={metrics["tp"]:6d}   FP={metrics["fp"]:6d}')
    print(f'    FN={metrics["fn"]:6d}   TN={metrics["tn"]:6d}')
    print(f'{sep}')
    print(f'  F1 ≥ {target_f1:.0%} target        : {status}')
    print(f'{sep}\n')


def plot_training_history(history: dict, output_dir: str) -> None:
    """Plot train/val loss and val F1/recall over epochs."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history['train_loss'], label='Train loss', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val loss',   color='firebrick')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['val_f1'], color='darkorange')
    axes[1].axhline(0.98, color='green', linestyle='--', label='Target F1=0.98')
    axes[1].set_title('Val F1 (FAULTY)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(history['lr_backbone'], label='Backbone LR', color='purple')
    axes[2].plot(history['lr_head'],     label='Head LR',     color='teal')
    axes[2].legend()

    axes[2].set_title('Learning rate'); axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

    # Phase bands — shade each phase a different colour
    phase_colours = {1:'#e8f4f8', 2:'#fef9e7', 3:'#eafaf1', 4:'#fdf2f8'}
    for ax in axes[:3]:
        prev = 0
        for phase, colour in phase_colours.items():
            end = [CFG['phase1_epochs'],
                   CFG['phase1_epochs'] + CFG['phase2_epochs'],
                   CFG['phase1_epochs'] + CFG['phase2_epochs'] + CFG['phase3_epochs'],
                   CFG['total_epochs']][phase - 1]
            ax.axvspan(prev, min(end, len(history['val_f1'])),
                       alpha=0.25, color=colour, label=f'Ph{phase}')
            prev = end

    plt.tight_layout()
    path = os.path.join(output_dir, 'training_history.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Training history saved → {path}')


def plot_evaluation_charts(
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> None:
    """Four evaluation charts saved as a single figure."""
    preds = (probs >= threshold).astype(int)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # ── 1. Confusion matrix ───────────────────────────────────────────────
    cm = confusion_matrix(labels, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
        xticklabels=['GOOD', 'FAULTY'], yticklabels=['GOOD', 'FAULTY'],
    )
    axes[0, 0].set_title('Confusion Matrix')
    axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

    # ── 2. Precision-Recall curve ─────────────────────────────────────────
    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    axes[0, 1].plot(rec_arr, prec_arr, lw=2, color='darkorange', label=f'AP={ap:.4f}')
    axes[0, 1].axvline(
        x=recall_score(labels, preds, zero_division=0),
        color='red', linestyle='--', lw=1.2, label=f'τ={threshold:.3f}'
    )
    axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
    axes[0, 1].set_title('Precision-Recall Curve')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # ── 3. ROC curve ─────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    axes[1, 0].plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC={auc:.4f}')
    axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[1, 0].set_xlabel('FPR'); axes[1, 0].set_ylabel('TPR')
    axes[1, 0].set_title('ROC Curve')
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    # ── 4. Confidence histogram ───────────────────────────────────────────
    bins = np.linspace(0, 1, 50)
    axes[1, 1].hist(probs[labels == 0], bins=bins, alpha=0.6, color='steelblue',  label='GOOD')
    axes[1, 1].hist(probs[labels == 1], bins=bins, alpha=0.6, color='firebrick',  label='FAULTY')
    axes[1, 1].axvline(threshold, color='black', linestyle='--', lw=1.5, label=f'τ={threshold:.3f}')
    axes[1, 1].set_xlabel('P(FAULTY)'); axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Confidence Score Distribution')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, 'evaluation_charts.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Evaluation charts saved → {path}')


def export_predictions(
    test_loader: DataLoader,
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> pd.DataFrame:
    """
    Export a CSV with one row per test image containing:
    - image path
    - P(FAULTY) probability
    - predicted class (0/1)
    - true class (0/1)
    - human-readable predicted/true labels
    - whether the prediction was correct
    """
    preds = (probs >= threshold).astype(int)
    df    = test_loader.dataset.df.copy()

    df['prob_faulty']    = probs
    df['pred_binary']    = preds
    df['true_binary']    = labels
    df['pred_class']     = ['FAULTY' if p == 1 else 'GOOD' for p in preds]
    df['true_class']     = ['FAULTY' if l == 1 else 'GOOD' for l in labels]
    df['correct']        = (preds == labels)
    df['threshold_used'] = threshold

    path = os.path.join(output_dir, 'predictions.csv')
    df.to_csv(path, index=False)
    print(f'Predictions exported → {path}  ({len(df)} rows)')
    return df


print('Evaluation and visualisation functions ready.')

Evaluation and visualisation functions ready.


In [ ]:
# os.path.exists("/kaggle/working/processed/annotations_processed.csv")
# df2 = pd.read_csv(CFG['annotation_csv'])
# df2.head()

# ---
## 13 — Main Execution

This cell ties everything together. Run it to go from raw CSV to a trained, calibrated, and evaluated model.

In [21]:
def run_full_pipeline(cfg=CFG, run_hparam_search=False):
    """
    Full end-to-end pipeline.

    Changes vs previous version
    ───────────────────────────
    1. pred_csv properly read from cfg (was undefined — would crash on run 2)
    2. build_prediction_based_weights called (was build_combined_weights with
       undefined pred_csv variable)
    3. mask_map passed to make_dataloaders (was built but never used)
    4. PREPROCESS_FIRST auto-detected via marker file (no hardcoded flag)
    5. make_dataloaders no longer receives rank/num_replicas (GPU-only now)
    """
    os.makedirs(cfg['output_dir'],    exist_ok=True)
    os.makedirs(cfg['processed_dir'], exist_ok=True)

    # # ── Step 1: COCO ROI map ───────────────────────────────────────────────
    print('\n═══ Step 1: Loading COCO annotations ===')
    roi_map, coco, id_to_filename, id_to_catname = load_coco_roi_map(
        './1st-krones-vision-ai-challenge/train_annotations.json',
        roi_category_id=cfg['roi_category_id'],
    )


    # ── Step 2: Load CSV ───────────────────────────────────────────────────
    print('\n═══ Step 2: Loading dataset ===')
    df = pd.read_csv(cfg['annotation_csv'])
    df = df.rename(columns={'image_id': cfg['image_col'], 'target': 'binary_label'})
    df['image_path'] = df['image_path'].apply(
        lambda x: os.path.join(cfg['image_dir'], x)
    )
    n_good   = (df['binary_label'] == 0).sum()
    n_faulty = (df['binary_label'] == 1).sum()
    log.info('Dataset: GOOD=%d (%.1f%%) | FAULTY=%d (%.1f%%) | Total=%d',
             n_good,   100*n_good/len(df),
             n_faulty, 100*n_faulty/len(df), len(df))

    # df = pd.read_csv('./1st-krones-vision-ai-challenge/annotations_processed.csv')

    
    # ── Step 3: Sample weights ────────────────────────────────────────────
    print('\n═══ Step 3: Building sample weights ===')
    pred_csv = cfg.get('predictions_csv', '')   # ← was undefined — fixed
    if pred_csv and Path(pred_csv).exists():
        print(f'  Mode: PREDICTION-BASED (FP/FN from {pred_csv})')
    else:
        print('  Mode: COCO FALLBACK (first run — no predictions.csv yet)')

    # build_prediction_based_weights replaces build_combined_weights
    df = build_prediction_based_weights(
        df, cfg, coco, id_to_filename, id_to_catname,
        image_col=cfg['image_col'],
    )

    defect_patches = build_defect_patch_library(
        df, coco, id_to_filename, id_to_catname, CFG)

    # # ── Step 4: GOOD feature mask map ─────────────────────────────────────
    print('\n═══ Step 4: Building GOOD feature mask map ===')
    # mask_map = build_good_feature_mask_map(coco, id_to_filename, id_to_catname)
    validate_roi_map(df, roi_map, image_col=cfg['image_col'])

    # ── Step 5: Pre-process ROIs (auto-detect) ────────────────────────────
    # Marker file records which input_size was used for saved crops.
    # Re-preprocessing runs automatically only when input_size changes
    # or crops have never been created. No more hardcoded True/False.
    processed_marker = Path(cfg['processed_dir']) / f'.done_{cfg["input_size"]}'
    PREPROCESS_FIRST = not processed_marker.exists()

    if PREPROCESS_FIRST:
        print(f'\n═══ Step 5: Pre-processing ROIs at {cfg["input_size"]}×{cfg["input_size"]} ===')
        df = preprocess_and_save_all(
            df, cfg['processed_dir'], cfg, roi_map=roi_map, n_jobs=4
        )
        # Update paths to point to processed directory
        df[cfg['image_col']] = df[cfg['image_col']].apply(
            lambda p: str(Path(cfg['processed_dir']) / Path(p).name)
        )
        cfg['annotation_csv'] = os.path.join(
            cfg['processed_dir'], 'annotations_processed.csv'
        )
        df.to_csv(cfg['annotation_csv'], index=False)
        processed_marker.touch()
        print(f'Pre-processing complete. Marker: {processed_marker}')
    else:
        # Point df paths to existing pre-cropped images
        df[cfg['image_col']] = df[cfg['image_col']].apply(
            lambda p: str(Path(cfg['processed_dir']) / Path(p).name)
        )
        print(f'Using existing {cfg["input_size"]}px crops from {cfg["processed_dir"]}')

    use_roi = False   # always False after preprocessing

    # print(df.head())
    # ── Step 6: DataLoaders ────────────────────────────────────────────────
    print('\n═══ Step 6: Building DataLoaders ===')
    train_loader, val_loader, test_loader, pos_weight = make_dataloaders(
        df, cfg,
        use_roi=use_roi,
        roi_map=roi_map,
        mask_map=None,   # ← was built but never passed before
        defect_patches=defect_patches,
    )

    # Sampler balance check
    good_c = faulty_c = 0
    for i, (imgs, lbls, _w) in enumerate(train_loader):
        good_c   += (lbls == 0).sum().item()
        faulty_c += (lbls == 1).sum().item()
        if i == 9: break
    total = good_c + faulty_c
    print(f'Sampler check — GOOD: {good_c} ({100*good_c/total:.1f}%)'
          f' | FAULTY: {faulty_c} ({100*faulty_c/total:.1f}%)')

    # ── Step 7 (optional): Optuna ──────────────────────────────────────────
    if run_hparam_search:
        print('\n═══ Step 7: Hyperparameter search (Optuna) ===')
        cfg, study = run_hyperparameter_search(train_loader, val_loader, cfg)
        print('Best hyperparameters applied.')

    # ── Step 8: Build model ────────────────────────────────────────────────
    print('\n═══ Step 8: Building model ===')
    model = build_model(cfg)

    # ── Step 9: Train ─────────────────────────────────────────────────────
    print('\n═══ Step 9: Training ===')
    history = train(model, train_loader, val_loader, cfg, DEVICE)
    plot_training_history(history, cfg['output_dir'])

    # ── Step 10: Load best checkpoint ─────────────────────────────────────
    print('\n═══ Step 10: Loading best checkpoint ===')
    ckpt = torch.load(cfg['checkpoint_path'], map_location=DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    print(f'Loaded epoch {ckpt["epoch"]} (val_f1={ckpt["val_f1"]:.4f})')
    model = model.to(DEVICE)

    # ── Step 11: Threshold calibration (val set) ───────────────────────────
    print('\n═══ Step 11: Calibrating threshold ===')
    val_metrics = evaluate(model, val_loader, DEVICE, cfg, threshold=0.5)
    threshold, val_calib = calibrate_threshold(
        val_metrics['probs'], val_metrics['labels'],
        min_recall=cfg['min_recall_faulty'],
    )
    print(f'Calibrated threshold: {threshold:.4f}')
    print_evaluation_report(val_calib, split='Validation (calibration)')

    # ── Step 12: Final evaluation (test set) ──────────────────────────────
    print('\n═══ Step 12: Final evaluation on test set ===')
    test_eval    = evaluate(model, test_loader, DEVICE, cfg, threshold=threshold)
    test_metrics = compute_metrics_at_threshold(
        test_eval['labels'], test_eval['probs'], threshold
    )
    print_evaluation_report(test_metrics, split='Test')
    preds = (test_eval['probs'] >= threshold).astype(int)
    print(classification_report(
        test_eval['labels'], preds,
        target_names=['GOOD', 'FAULTY'], digits=4,
    ))

    # ── Step 13: Save outputs ─────────────────────────────────────────────
    print('\n═══ Step 13: Saving outputs ===')
    plot_evaluation_charts(
        test_eval['probs'], test_eval['labels'],
        threshold, cfg['output_dir'],
    )
    pred_df = export_predictions(
        test_loader, test_eval['probs'], test_eval['labels'],
        threshold, cfg['output_dir'],
    )
    metrics_path = os.path.join(cfg['output_dir'], 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(
            {k: round(float(v), 6) for k, v in test_metrics.items()},
            f, indent=2,
        )
    print(f'Metrics JSON → {metrics_path}')
    print('\n═══ PIPELINE COMPLETE ===')
    return model, threshold, test_metrics, pred_df
    # return test_loader


# ── LAUNCH ────────────────────────────────────────────────────────────────
model, threshold, metrics, predictions = run_full_pipeline(
    cfg=CFG,
    run_hparam_search=False,
)
# test_loader = run_full_pipeline(
#     cfg=CFG,
#     run_hparam_search=False,
# )


═══ Step 1: Loading COCO annotations ===


INFO | COCO ROI map — 35342 images mapped | category_id=22 | skipped=0
INFO | Dataset: GOOD=14729 (41.7%) | FAULTY=20613 (58.3%) | Total=35342
INFO | Prediction-based weights | threshold=0.2314 | FP→upweight ×4.0: 398 bottles | FN→upweight ×1.5: 23 bottles
INFO | Weight dist — FP (×4.0): 398 | FN (×1.5): 23 | normal: 34921



═══ Step 2: Loading dataset ===

═══ Step 3: Building sample weights ===
  Mode: PREDICTION-BASED (FP/FN from ./1st-krones-vision-ai-challenge/predictions.csv)


INFO | Defect patch library: 1 contamination_dark patches extracted
INFO | ROI validation passed — all 35342 images have COCO annotations.



═══ Step 4: Building GOOD feature mask map ===


...


═══ Step 9: Training ===


INFO | Ep   1 [ph1 | w=OFF] | loss 0.0454→0.0230 | F1 0.8138 | rec 0.7196 | lr_bb 5.0e-04 lr_hd 5.0e-04
INFO |   ↑ New best val_f1=0.8138 saved.
INFO | Ep   2 [ph1 | w=OFF] | loss 0.0300→0.0218 | F1 0.8282 | rec 0.7384 | lr_bb 7.5e-04 lr_hd 7.5e-04
INFO |   ↑ New best val_f1=0.8282 saved.
INFO | Ep   3 [ph1 | w=OFF] | loss 0.0253→0.0202 | F1 0.8672 | rec 0.8205 | lr_bb 1.0e-03 lr_hd 1.0e-03
INFO |   ↑ New best val_f1=0.8672 saved.
INFO | Ep   4 [ph1 | w=OFF] | loss 0.0240→0.0282 | F1 0.7418 | rec 0.5941 | lr_bb 1.0e-03 lr_hd 1.0e-03
INFO | Ep   5 [ph1 | w=OFF] | loss 0.0235→0.0197 | F1 0.8432 | rec 0.7503 | lr_bb 9.6e-04 lr_hd 9.6e-04
INFO | Ep   6 [ph1 | w=OFF] | loss 0.0225→0.0193 | F1 0.8791 | rec 0.8422 | lr_bb 8.5e-04 lr_hd 8.5e-04
INFO |   ↑ New best val_f1=0.8791 saved.
INFO | Ep   7 [ph1 | w=OFF] | loss 0.0221→0.0188 | F1 0.8602 | rec 0.7814 | lr_bb 6.9e-04 lr_hd 6.9e-04
INFO | Ep   8 [ph1 | w=OFF] | loss 0.0223→0.0181 | F1 0.8721 | rec 0.8017 | lr_bb 5.0e-04 lr_hd 5.0e-04
INFO

Training history saved → ./1st-krones-vision-ai-challenge/outputs/training_history.png

═══ Step 10: Loading best checkpoint ===
Loaded epoch 32 (val_f1=0.9157)

═══ Step 11: Calibrating threshold ===


INFO | Calibrated threshold=0.2407 | F1(FAULTY)=0.8443 | Recall=0.9903 | Prec=0.7359


Calibrated threshold: 0.2407

──────────────────────────────────────────────────────────────
  BOTTLE INSPECTION — VALIDATION (CALIBRATION) SET EVALUATION
──────────────────────────────────────────────────────────────
  Decision threshold   : 0.2407
──────────────────────────────────────────────────────────────
  F1 (FAULTY class)    : 0.8443   ← primary KPI
  Recall    (FAULTY)   : 0.9903   ← safety metric
  Precision (FAULTY)   : 0.7359
  F1 (macro)           : 0.7536
  Accuracy             : 0.7870
  ROC-AUC              : 0.9745
  PR-AUC               : 0.9823
──────────────────────────────────────────────────────────────
  Confusion matrix:
    TP=  3062   FP=  1099
    FN=    30   TN=  1110
──────────────────────────────────────────────────────────────
  F1 ≥ 98% target        : ✗ BELOW TARGET
──────────────────────────────────────────────────────────────


═══ Step 12: Final evaluation on test set ===

──────────────────────────────────────────────────────────────
  BOTTLE INSPE

In [ ]:
# replace the image path /kaggle/working/processed with .1st-krones-vision-ai-challenge/processed in the predictions dataframe for display
# df = pd.read_csv('./1st-krones-vision-ai-challenge/predictions.csv')
# df['image_path'] = df['image_path'].str.replace('/kaggle/working/processed', './1st-krones-vision-ai-challenge/processed')

# # save the modified dataframe to a new csv file
# df.to_csv('./1st-krones-vision-ai-challenge/predictions.csv', index=False)

In [ ]:
# import shutil
# shutil.rmtree("/kaggle/working/")
# os.remove('/kaggle/working/submission.csv')

In [ ]:
df = pd.read_csv('/kaggle/working/processed/annotations_processed.csv')
df.head()
# print(CFG['annotation_csv'])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SUBMISSION — uses test annotations JSON for ROI cropping
# ═══════════════════════════════════════════════════════════════════════════

import glob

TEST_IMAGE_DIR       = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images'
TEST_ANNOTATIONS_JSON = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_annotations_roi_only.json'
OUTPUT_FILE          = 'submission.csv'


def load_test_roi_map(
    annotation_path: str,
    roi_category_id: int = 22,
) -> Dict[str, Tuple[int, int, int]]:
    """
    Build filename → (cx, cy, radius) lookup from the test annotations JSON.

    The test JSON has the same COCO structure as the training one:
      "images":      [{"id": 1, "file_name": "...", "height": 1024, "width": 1280}]
      "annotations": [{"image_id": 1, "category_id": 22, "bbox": [x,y,w,h]}]

    If the test JSON has no annotations section (annotations-free test set),
    the function returns an empty dict and prediction falls back to the
    fixed cx/cy/radius from CFG.
    """
    with open(annotation_path, 'r') as f:
        data = json.load(f)

    # image_id → basename
    id_to_filename = {
        img['id']: Path(img['file_name']).name
        for img in data.get('images', [])
    }

    # If no annotations key, return empty (will use fixed fallback)
    if 'annotations' not in data or len(data['annotations']) == 0:
        log.warning(
            'Test annotations JSON has no annotations — will use fixed '
            'cx/cy/radius fallback for all test images.'
        )
        return {}

    roi_map: Dict[str, Tuple[int, int, int]] = {}
    skipped = 0

    for ann in data['annotations']:
        if ann.get('category_id') != roi_category_id:
            continue

        filename = id_to_filename.get(ann['image_id'])
        if filename is None:
            continue

        # Prefer segmentation polygon for accurate circle fit
        if ann.get('segmentation') and len(ann['segmentation']) > 0:
            try:
                cx, cy, radius = _circle_from_segmentation(ann['segmentation'])
            except Exception:
                if ann.get('bbox') and len(ann['bbox']) == 4:
                    cx, cy, radius = _circle_from_bbox(ann['bbox'])
                else:
                    skipped += 1
                    continue
        elif ann.get('bbox') and len(ann['bbox']) == 4:
            cx, cy, radius = _circle_from_bbox(ann['bbox'])
        else:
            skipped += 1
            continue

        roi_map[filename] = (cx, cy, radius)

    log.info(
        'Test ROI map — %d images mapped | category_id=%d | skipped=%d',
        len(roi_map), roi_category_id, skipped,
    )
    return roi_map


def predict_test_image(
    image_path: str,
    model:      nn.Module,
    threshold:  float,
    cfg:        dict,
    device,
    roi_map:    Optional[Dict[str, Tuple[int, int, int]]] = None,
    show:       bool = False,
) -> int:
    """
    Predict GOOD (0) or FAULTY (1) for a single test image.

    Changes vs original
    ───────────────────
    1. Accepts roi_map — uses COCO-derived circle if available,
       falls back to fixed cx/cy/radius if not.
    2. device passed as torch.device, not str — avoids autocast issue.
    3. Threshold must be the CALIBRATED value from training (0.162),
       NOT 0.4 (the old hardcoded value missed ~900 FAULTY bottles).
    """
    # Step 1: ROI extraction using COCO annotation or fixed fallback
    pil_img = extract_roi_from_path(str(image_path), cfg, roi_map=roi_map)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(Image.open(image_path))
        axes[0].set_title('Raw'); axes[0].axis('off')
        axes[1].imshow(pil_img)
        axes[1].set_title(f'ROI crop → model'); axes[1].axis('off')
        plt.tight_layout(); plt.show()

    # Step 2: Normalise and convert to tensor
    transform  = build_eval_transforms(cfg)
    np_img     = np.array(pil_img)
    tensor     = transform(image=np_img)['image'].unsqueeze(0).to(device)

    # Step 3: Inference
    model.eval()
    with torch.no_grad():
        with torch.amp.autocast(
            device_type=device.type if hasattr(device, 'type') else str(device),
            enabled=cfg.get('amp', True) and 'cuda' in str(device),
        ):
            logit = model(tensor).squeeze()

    prob_faulty = float(torch.sigmoid(logit).cpu())

    # Step 4: Decision using calibrated threshold
    return int(prob_faulty >= threshold)


def make_submission(
    calibrated_threshold: Optional[float] = None,
    model_instance:       Optional[nn.Module] = None,
):
    """
    Generate submission.csv for the test set.

    Parameters
    ----------
    calibrated_threshold : the threshold from calibrate_threshold() after training.
                           If None, loads from the saved checkpoint's cfg
                           or falls back to CFG['default_threshold'].
                           DO NOT use 0.4 — the calibrated value is ~0.162.
    model_instance       : if a trained model is already in memory, pass it
                           to skip reloading from disk.
    """
    SUBMISSION_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Submission device: {SUBMISSION_DEVICE}')

    # ── Load model ────────────────────────────────────────────────────────
    if model_instance is not None:
        submit_model = model_instance
        print('Using in-memory trained model.')
    else:
        print(f'Loading model from {CFG["checkpoint_path"]} ...')
        submit_model = build_model(CFG)
        ckpt = torch.load(CFG['checkpoint_path'], map_location=SUBMISSION_DEVICE)
        submit_model.load_state_dict(ckpt['state_dict'])
        print(f'Loaded checkpoint from epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_f1"]:.4f})')

    submit_model.to(SUBMISSION_DEVICE)
    submit_model.eval()

    # ── Determine threshold ───────────────────────────────────────────────
    if calibrated_threshold is not None:
        tau = calibrated_threshold
    else:
        # Try to read from checkpoint cfg, then fall back to CFG default
        ckpt_loaded = torch.load(CFG['checkpoint_path'],
                                  map_location='cpu')
        tau = ckpt_loaded.get('cfg', {}).get('default_threshold',
                                              CFG['default_threshold'])
        log.warning(
            'No calibrated_threshold provided. Using %.4f. '
            'Pass the threshold from calibrate_threshold() for best results.',
            tau,
        )

    print(f'Decision threshold: {tau:.4f}')
    print(f'NOTE: this must be the CALIBRATED threshold (~0.162), '
          f'not 0.4 or 0.5.')

    # ── Load test ROI annotations ─────────────────────────────────────────
    test_roi_map = {}
    if Path(TEST_ANNOTATIONS_JSON).exists():
        print(f'Loading test ROI annotations from {TEST_ANNOTATIONS_JSON} ...')
        test_roi_map = load_test_roi_map(
            TEST_ANNOTATIONS_JSON,
            roi_category_id=CFG['roi_category_id'],
        )
        print(f'Test ROI map: {len(test_roi_map)} images with COCO annotations.')
    else:
        print(f'Test annotations JSON not found at {TEST_ANNOTATIONS_JSON}.')
        print('Falling back to fixed cx/cy/radius for all test images.')

    # ── Run predictions ───────────────────────────────────────────────────
    image_paths = sorted(glob.glob(os.path.join(TEST_IMAGE_DIR, '*.png')))
    print(f'Found {len(image_paths)} test images.')

    if len(image_paths) == 0:
        raise FileNotFoundError(
            f'No PNG images found in {TEST_IMAGE_DIR}. '
            f'Check the TEST_IMAGE_DIR path.'
        )

    # Count how many test images have COCO annotations
    n_with_roi = sum(
        1 for p in image_paths if Path(p).name in test_roi_map
    )
    n_fallback  = len(image_paths) - n_with_roi
    print(f'  → Using COCO ROI:      {n_with_roi} images')
    print(f'  → Using fixed fallback: {n_fallback} images')

    submissions = []
    for path in tqdm(image_paths, desc='Predicting test images'):
        target = predict_test_image(
            image_path=path,
            model=submit_model,
            threshold=tau,
            cfg=CFG,
            device=SUBMISSION_DEVICE,
            roi_map=test_roi_map if test_roi_map else None,
            show=False,
        )
        submissions.append({
            'image_id': os.path.basename(path),
            'target':   target,
        })

    # ── Save ─────────────────────────────────────────────────────────────
    df_sub = pd.DataFrame(submissions)
    df_sub.to_csv(OUTPUT_FILE, index=False)

    n_faulty = (df_sub['target'] == 1).sum()
    n_good   = (df_sub['target'] == 0).sum()
    print(f'\nSubmission saved → {OUTPUT_FILE}')
    print(f'  GOOD:   {n_good}  ({100*n_good/len(df_sub):.1f}%)')
    print(f'  FAULTY: {n_faulty} ({100*n_faulty/len(df_sub):.1f}%)')
    print(f'  Total:  {len(df_sub)}')
    return df_sub


# ── Run ───────────────────────────────────────────────────────────────────
# Pass the calibrated threshold from the training run.
# It was stored in the variable `threshold` after run_full_pipeline().
# If you restarted the kernel, re-run calibration or use the value from metrics.json.

CALIBRATED_THRESHOLD = 0.162476   # from metrics.json — update if you retrain

submission_df = make_submission(
    calibrated_threshold=CALIBRATED_THRESHOLD,
    model_instance=model,   # pass the in-memory model to avoid reloading
)

In [ ]:
# import pandas as pd
# import glob
# import os
# import torch

# # Settings
# TEST_IMAGE_DIR = r'/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images'
# OUTPUT_FILE = 'submission.csv'

# def predict_test_image(
#     image_path: str,
#     model: nn.Module,
#     threshold: float,
#     cfg: dict,
#     device: torch.device,
#     show: bool = True,
# ):
#       # Step 1: ROI extraction
#     pil_img = extract_roi_from_path(image_path, cfg)

#     if show:
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))
#         axes[0].imshow(Image.open(image_path))
#         axes[0].set_title('Raw image'); axes[0].axis('off')
#         axes[1].imshow(pil_img)
#         axes[1].set_title('ROI crop (input to model)'); axes[1].axis('off')
#         plt.tight_layout(); plt.show()

#     # Step 2: Normalise and convert to tensor
#     transform = build_eval_transforms(cfg)
#     np_img  = np.array(pil_img)
#     tensor  = transform(image=np_img)['image'].unsqueeze(0).to(device)  # [1,3,H,W]

#     # Step 3: Inference
#     with torch.no_grad():
#         with torch.amp.autocast(device_type = 'cuda'):
#             logit = model(tensor).squeeze()  # scalar
#     prob_faulty = float(torch.sigmoid(logit).cpu())

#     # Step 4: Decision
#     prediction = int(prob_faulty >= threshold) # 1 for FAULTY, 0 for GOOD
#     # label      = 'FAULTY' if prediction == 1 else 'GOOD'
#     return prediction


# def make_submission():
#     # 1. Use glob to find all images (handles pathing easily)
#     image_paths = glob.glob(os.path.join(TEST_IMAGE_DIR, "*.png"))

#     # 2. Extract filenames and generate predictions
#     # Replace the '0' with your actual model prediction output
#     submissions = []
#     # load the trained model (assuming it's saved as 'final_model.pth' in the output directory)
#     # ml_model = torch.load(os.path.join('/kaggle/working/final_model.pth'), map_location=DEVICE)
#     # ml_model.eval()  # set model to evaluation mode

#     model_path = '/kaggle/working/outputs/best_model.pt'  # Or your path
#     DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
#     print(DEVICE)
#     # Instantiate the model
#     model = build_model(CFG)
#     # Load the state_dict
#     state_dict = torch.load(model_path, map_location=DEVICE)
#     model.load_state_dict(state_dict["state_dict"])
#     model.to(DEVICE)
#     model.eval() 

    
#     for path in tqdm(image_paths, desc="Predicting each image"):
#         # Perform inference on the image and get the predicted label (0 or 1)

#         target =  predict_test_image(
#             image_path=path,
#             model=model,
#             threshold=0.4,
#             cfg=CFG,
#             device=DEVICE,
#             show=False,  # Set to True if you want to visualize each prediction
#         )
#         submissions.append({
#             "image_id": os.path.basename(path),
#             "target": target  
#         })

#     # 3. Save to CSV
#     df = pd.DataFrame(submissions)
#     df.to_csv(OUTPUT_FILE, index=False)

#     print(f"Created {OUTPUT_FILE} with {len(df)} rows.")

# make_submission()


In [ ]:
@torch.no_grad()
def evaluate_with_tta(model, loader, cfg, device, n_aug=8, threshold=0.5):
    """
    Test-Time Augmentation: run each image through N augmented versions
    and average the predicted probabilities before applying threshold.
    
    Why it helps for the 119 hard bottles:
    - These bottles score 0.23-0.43 because contamination_dark is partially
      occluded or outweighed by water_drop in the standard crop.
    - Random crops/rotations sometimes frame the contamination region better.
    - Averaging 8 views is statistically more likely to include views where
      the defect is prominent → average score shifts upward.
    """
    aug_transform  = build_train_transforms(cfg)   # stochastic
    eval_transform = build_eval_transforms(cfg)     # deterministic

    model.eval()
    all_probs, all_labels = [], []

    for images_orig, labels, _w in tqdm(loader, desc=f'TTA ×{n_aug}'):
        batch_probs = []

        # View 0: no augmentation (deterministic baseline)
        with torch.amp.autocast(device_type=device.type,
                                enabled=cfg['amp'] and device.type == 'cuda'):
            logits = model(images_orig.to(device)).squeeze(1)
        batch_probs.append(torch.sigmoid(logits).cpu().numpy())

        # Views 1..n_aug-1: random augmentations
        for _ in range(n_aug - 1):
            aug_imgs = []
            for img_tensor in images_orig:
                # Tensor → numpy → augment → tensor
                np_img   = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
                aug_tens = aug_transform(image=np_img)['image']
                aug_imgs.append(aug_tens)
            aug_batch = torch.stack(aug_imgs).to(device)
            with torch.amp.autocast(device_type=device.type,
                                    enabled=cfg['amp'] and device.type == 'cuda'):
                logits = model(aug_batch).squeeze(1)
            batch_probs.append(torch.sigmoid(logits).cpu().numpy())

        # Average across all views
        avg_probs = np.mean(batch_probs, axis=0)
        all_probs.extend(avg_probs.tolist())
        all_labels.extend(labels.numpy().astype(int).tolist())

    probs_arr  = np.array(all_probs,  dtype=np.float32)
    labels_arr = np.array(all_labels, dtype=np.int32)

    # Re-calibrate threshold on the TTA probabilities
    tta_threshold, tta_metrics = calibrate_threshold(
        probs_arr, labels_arr, min_recall=cfg['min_recall_faulty']
    )
    return probs_arr, labels_arr, tta_threshold, tta_metrics


# Run immediately on current model — no retraining needed
tta_probs, tta_labels, tta_tau, tta_metrics = evaluate_with_tta(
    model, test_loader, CFG, DEVICE, n_aug=8, threshold=CFG['default_threshold']
)
test_tta = compute_metrics_at_threshold(tta_labels, tta_probs, tta_tau)
print_evaluation_report(test_tta, split='Test (TTA ×8)')

In [33]:
# !pip install grad-cam

In [37]:
# pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def inspect_hard_faulty_gradcam(model, pred_csv, cfg, device, n_show=10):
    """
    Visualise what the model attends to for hard FAULTY bottles.
    Uses GradCAM on the CBAM attention module (spatial feature map layer).
    """
    preds = pd.read_csv(pred_csv)
    hard  = preds[
        (preds['true_binary'] == 1) &
        (preds['prob_faulty'] >= 0.23) &
        (preds['prob_faulty'] <= 0.43)
    ].sort_values('prob_faulty').head(n_show)

    # GradCAM requires a layer that outputs 4D spatial feature maps.
    # CBAM modules work on [B, C, H, W] and are ideal for visualization.
    if hasattr(model, 'cbams'):
        # BottleDetector with multi-scale: target the last CBAM (stage 4)
        target_layer = model.cbams[-1]
    elif hasattr(model, 'cbam'):
        # BottleClassifier with single-scale: target the spatial attention
        target_layer = model.cbam
    else:
        raise ValueError('Model must have cbam or cbams attribute')
    
    cam = GradCAM(model=model, target_layers=[target_layer])

    fig, axes = plt.subplots(n_show, 2, figsize=(8, n_show * 3))
    transform = build_eval_transforms(cfg)

    for i, (_, row) in enumerate(hard.iterrows()):
        pil    = Image.open(row['image_path']).convert('RGB')
        np_img = np.array(pil)
        tensor = transform(image=np_img)['image'].unsqueeze(0).to(device)

        grayscale_cam = cam(input_tensor=tensor)[0]
        vis = show_cam_on_image(
            np.array(pil.resize((cfg['input_size'], cfg['input_size']))) / 255.0,
            grayscale_cam, use_rgb=True
        )
        axes[i, 0].imshow(pil.resize((cfg['input_size'], cfg['input_size'])))
        axes[i, 0].set_title(f'P(FAULTY)={row["prob_faulty"]:.3f}')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(vis)
        axes[i, 1].set_title('GradCAM attention')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(cfg['output_dir'], 'gradcam_hard_faulty.png'), dpi=120)
    plt.show()

inspect_hard_faulty_gradcam(model, CFG['predictions_csv'], CFG, DEVICE)

In [38]:
# Load predictions from the current run
pred_df = pd.read_csv('./1st-krones-vision-ai-challenge/outputs/predictions.csv')
probs  = pred_df['prob_faulty'].values
labels = pred_df['true_binary'].values

print(f"{'τ':>6}  {'F1':>6}  {'Prec':>6}  {'Rec':>6}")
for tau in np.arange(0.05, 0.95, 0.01):
    preds  = (probs >= tau).astype(int)
    f1_val = f1_score(labels, preds, zero_division=0)
    prec   = precision_score(labels, preds, zero_division=0)
    rec    = recall_score(labels, preds, zero_division=0)
    marker = ' ← best F1' if f1_val == max(
        f1_score(labels, (probs >= t).astype(int), zero_division=0)
        for t in np.arange(0.05, 0.95, 0.01)
    ) else ''
    if rec >= 0.95:  # only show rows with reasonable recall
        print(f'{tau:6.2f}  {f1_val:6.4f}  {prec:6.4f}  {rec:6.4f}{marker}')

     τ      F1    Prec     Rec
  0.05  0.7409  0.5884  1.0000
  0.06  0.7475  0.5968  1.0000
  0.07  0.7575  0.6096  1.0000
  0.08  0.7701  0.6262  0.9997
  0.09  0.7855  0.6470  0.9994
  0.10  0.8045  0.6732  0.9994
  0.11  0.8229  0.6994  0.9994
  0.12  0.8404  0.7254  0.9987
  0.13  0.8574  0.7512  0.9987
  0.14  0.8709  0.7723  0.9984
  0.15  0.8843  0.7941  0.9977
  0.16  0.8958  0.8132  0.9971
  0.17  0.9042  0.8272  0.9971
  0.18  0.9126  0.8420  0.9961
  0.19  0.9190  0.8531  0.9958
  0.20  0.9242  0.8635  0.9942
  0.21  0.9281  0.8703  0.9942
  0.22  0.9309  0.8759  0.9932
  0.23  0.9351  0.8839  0.9926
  0.24  0.9378  0.8890  0.9922
  0.25  0.9428  0.8981  0.9922
  0.26  0.9445  0.9025  0.9906
  0.27  0.9466  0.9072  0.9897
  0.28  0.9482  0.9104  0.9893
  0.29  0.9503  0.9147  0.9887
  0.30  0.9521  0.9190  0.9877
  0.31  0.9540  0.9232  0.9871
  0.32  0.9546  0.9251  0.9861
  0.33  0.9556  0.9289  0.9838
  0.34  0.9564  0.9321  0.9819
  0.35  0.9568  0.9335  0.9812
  0.36  

In [40]:
import json
from collections import defaultdict

# ── Load COCO annotations ─────────────────────────────────────────────────
with open('./1st-krones-vision-ai-challenge/train_annotations.json') as f:
    coco = json.load(f)

# Build lookup maps
id_to_filename  = {img['id']: img['file_name'] for img in coco['images']}
id_to_catname   = {cat['id']: cat['name']      for cat in coco['categories']}

print('Categories in COCO file:')
for cat in coco['categories']:
    print(f"  id={cat['id']:3d}  name={cat['name']}")

Categories in COCO file:
  id=  1  name=Air bubble
  id=  2  name=Break / Crack
  id=  3  name=Chip
  id=  4  name=Circlip
  id=  5  name=Contamination dark
  id=  6  name=Contamination light
  id=  7  name=Crown cap
  id=  8  name=Embossing
  id=  9  name=Foam residue
  id= 10  name=Foil / Semitransparent
  id= 11  name=Foreign object - manual cleaning
  id= 12  name=Foreign object - washing machine
  id= 13  name=Glass imperfection
  id= 14  name=Glass shard
  id= 15  name=Insect
  id= 16  name=Label
  id= 17  name=Liquid
  id= 18  name=Mold
  id= 19  name=No base visible
  id= 20  name=No fault
  id= 21  name=Paint residue
  id= 22  name=Roi
  id= 23  name=Scuffing
  id= 24  name=Scuffing heavy
  id= 25  name=Straw
  id= 26  name=Water drop
  id= 27  name=Yeast residue


In [41]:
# ── Build image → list of defects map ────────────────────────────────────
# Each image can have MULTIPLE annotations (multiple defects)
image_defects = defaultdict(list)

for ann in tqdm(coco['annotations']):
    fname    = id_to_filename.get(ann['image_id'], '')
    basename = os.path.basename(fname)
    cat_name = id_to_catname.get(ann['category_id'], 'unknown')

    # Area: use annotation area field if present, else compute from bbox
    area = ann.get('area', None)
    if area is None and ann.get('bbox'):
        x, y, w, h = ann['bbox']
        area = w * h

    image_defects[basename].append({
        'label':       cat_name,
        'area':        area,
        'category_id': ann['category_id'],
    })

print(f'Total images with annotations: {len(image_defects)}')
print(f'Sample entry:')
sample_key = list(image_defects.keys())[0]
print(f'  {sample_key}: {image_defects[sample_key]}')

100%|██████████| 134471/134471 [00:00<00:00, 620083.28it/s]

Total images with annotations: 35342
Sample entry:
  ad0f5a12-93fe-4ebb-9aae-ef12ef5246f8_000000000001.png: [{'label': 'No fault', 'area': 251001.0, 'category_id': 20}, {'label': 'Roi', 'area': 261121.0, 'category_id': 22}]


In [42]:
# ── Join with predictions CSV ─────────────────────────────────────────────
pred_df = pd.read_csv('./1st-krones-vision-ai-challenge/outputs/predictions.csv')
pred_df['filename'] = pred_df['image_path'].apply(lambda x: os.path.basename(str(x)))

# Expand: one row per defect per image (since images can have multiple)
rows = []
for _, row in pred_df.iterrows():
    defects = image_defects.get(row['filename'], [])
    if not defects:
        # Image had no COCO annotation — shouldn't happen for FAULTY
        rows.append({**row.to_dict(), 'label': 'no_annotation', 'area': None})
    else:
        for defect in defects:
            rows.append({
                **row.to_dict(),
                'label':  defect['label'],
                'area':   defect['area'],
            })

expanded_df = pd.DataFrame(rows)

# ── Analyse hard FAULTY cluster ────────────────────────────────────────────
hard_faulty = expanded_df[
    (expanded_df['true_binary'] == 1) &
    (expanded_df['prob_faulty'] >= 0.169) &   # above current threshold
    (expanded_df['prob_faulty'] <= 0.45)       # below confident FAULTY zone
].copy()

# Deduplicate: for images with multiple defects, show all defect labels
print(f'\n=== Hard FAULTY images: {hard_faulty["filename"].nunique()} unique images ===')
print(f'=== Total defect annotations in those images: {len(hard_faulty)} ===')
print(f'\nLabel distribution (all defects on hard FAULTY images):')
print(hard_faulty['label'].value_counts())

CONDITIONAL_THRESHOLDS = {
    'air_bubble':            500,
    'chip':                  200,
    'contamination_light':   180,
    'glass_imperfection':    100,
    'scuffing':           75_000,
    'scuffing_heavy':      1_200,
}

# For conditional defects: how close to threshold?
print(f'\n=== Conditional defects — proximity to area threshold ===')
for label, thresh in CONDITIONAL_THRESHOLDS.items():
    subset = hard_faulty[
        (hard_faulty['label'] == label) &
        (hard_faulty['area'].notna())
    ]
    if len(subset) == 0:
        continue
    margin = subset['area'] - thresh
    print(f'  {label:25s} threshold={thresh:6d}px | n={len(subset):3d} | '
          f'margin above threshold → '
          f'min={margin.min():.0f}  mean={margin.mean():.0f}  max={margin.max():.0f}px')


=== Hard FAULTY images: 100 unique images ===
=== Total defect annotations in those images: 664 ===

Label distribution (all defects on hard FAULTY images):
label
Scuffing               147
Water drop             134
Roi                    100
Contamination dark      91
Foam residue            82
Scuffing heavy          56
Mold                    20
Contamination light     19
Air bubble              10
Chip                     2
Label                    1
Break / Crack            1
Liquid                   1
Name: count, dtype: int64

=== Conditional defects — proximity to area threshold ===


In [44]:
pred_df = pd.read_csv('./1st-krones-vision-ai-challenge/outputs/predictions.csv')
probs   = pred_df['prob_faulty'].values
labels  = pred_df['true_binary'].values

print(f"{'τ':>6}  {'F1':>7}  {'Prec':>7}  {'Rec':>7}  {'FP':>6}  {'FN':>6}")
best_f1 = 0
best_tau = 0
for tau in np.arange(0.05, 0.99, 0.01):
    preds = (probs >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0,1]).ravel()
    f1_val = f1_score(labels, preds, zero_division=0)
    prec   = precision_score(labels, preds, zero_division=0)
    rec    = recall_score(labels, preds, zero_division=0)
    if f1_val > best_f1:
        best_f1  = f1_val
        best_tau = tau
    print(f'{tau:6.2f}  {f1_val:7.4f}  {prec:7.4f}  {rec:7.4f}  {fp:6.0f}  {fn:6.0f}')

print(f'\nAbsolute ceiling: F1={best_f1:.4f} at τ={best_tau:.2f}')
print(f'Gap to F1=0.98: {0.98 - best_f1:.4f}')

     τ       F1     Prec      Rec      FP      FN
  0.05   0.7409   0.5884   1.0000    2163       0
  0.06   0.7475   0.5968   1.0000    2089       0
  0.07   0.7575   0.6096   1.0000    1980       0
  0.08   0.7701   0.6262   0.9997    1845       1
  0.09   0.7855   0.6470   0.9994    1686       2
  0.10   0.8045   0.6732   0.9994    1500       2
  0.11   0.8229   0.6994   0.9994    1328       2
  0.12   0.8404   0.7254   0.9987    1169       4
  0.13   0.8574   0.7512   0.9987    1023       4
  0.14   0.8709   0.7723   0.9984     910       5
  0.15   0.8843   0.7941   0.9977     800       7
  0.16   0.8958   0.8132   0.9971     708       9
  0.17   0.9042   0.8272   0.9971     644       9
  0.18   0.9126   0.8420   0.9961     578      12
  0.19   0.9190   0.8531   0.9958     530      13
  0.20   0.9242   0.8635   0.9942     486      18
  0.21   0.9281   0.8703   0.9942     458      18
  0.22   0.9309   0.8759   0.9932     435      21
  0.23   0.9351   0.8839   0.9926     403      23


---
## 14 — Single Image Inference

Once the model is trained, use this function to predict on a single bottle image.

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════
# # SINGLE IMAGE INFERENCE
# # ═══════════════════════════════════════════════════════════════════════════

# def predict_single_image(
#     image_path: str,
#     model: nn.Module,
#     threshold: float,
#     cfg: dict,
#     device: torch.device,
#     show: bool = True,
# ) -> dict:
#     """
#     Predict whether a single bottle image is GOOD or FAULTY.

#     Runs the full pipeline:
#     1. ROI extraction (Hough circle → crop → CLAHE → resize)
#     2. Normalisation
#     3. Model inference
#     4. Threshold comparison

#     Parameters
#     ----------
#     image_path : path to a raw bottle base image
#     model      : trained BottleClassifier
#     threshold  : calibrated decision threshold
#     cfg        : global CFG
#     device     : inference device
#     show       : whether to display the processed image in the notebook

#     Returns
#     -------
#     dict with keys: prob_faulty, prediction (0/1), label (GOOD/FAULTY), confident
#     """
#     model.eval()

#     # Step 1: ROI extraction
#     pil_img = extract_roi_from_path(image_path, cfg)

#     if show:
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))
#         axes[0].imshow(Image.open(image_path))
#         axes[0].set_title('Raw image'); axes[0].axis('off')
#         axes[1].imshow(pil_img)
#         axes[1].set_title('ROI crop (input to model)'); axes[1].axis('off')
#         plt.tight_layout(); plt.show()

#     # Step 2: Normalise and convert to tensor
#     transform = build_eval_transforms(cfg)
#     np_img  = np.array(pil_img)
#     tensor  = transform(image=np_img)['image'].unsqueeze(0).to(device)  # [1,3,H,W]

#     # Step 3: Inference
#     with torch.no_grad():
#         with torch.amp.autocast(device_type = 'cuda'):
#             logit = model(tensor).squeeze()  # scalar
#     prob_faulty = float(torch.sigmoid(logit).cpu())

#     # Step 4: Decision
#     prediction = int(prob_faulty >= threshold)
#     label      = 'FAULTY' if prediction == 1 else 'GOOD'

#     # Confidence: how far from the threshold?
#     confident  = abs(prob_faulty - threshold) > 0.2

#     result = {
#         'prob_faulty': round(prob_faulty, 4),
#         'prediction':  prediction,
#         'label':       label,
#         'confident':   confident,
#         'threshold':   threshold,
#     }

#     print(f'\n  Image      : {os.path.basename(image_path)}')
#     print(f'  P(FAULTY)  : {prob_faulty:.4f}')
#     print(f'  Decision   : {label} (threshold={threshold:.4f})')
#     print(f'  Confident  : {"Yes" if confident else "Borderline"}')

#     return result


# # ── Example usage (replace with an actual image path from your dataset) ───
# # result = predict_single_image(
# #     image_path='/kaggle/input/your-dataset/bottle_001.png',
# #     model=model,
# #     threshold=threshold,
# #     cfg=CFG,
# #     device=DEVICE,
# #     show=True,
# # )
# # print('Single-image inference function ready.')